<h1><center>Laboratorio 10: Radio fcFM, ¿se puede conocer el ánimo de una canción por su letra? 🎧</center></h1>

<center><strong>IA7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla e Ignacio Núñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Cornejo, Ignacio Reyes

### Equipo

- Nombre de estudiante 1: Cristóbal Lioi
- Nombre de estudiante 2: Julio Badilla

### Reglas

- Trabajar en equipos de hasta dos personas.
- Las letras tienen derechos de autor y se entregan solo para este curso: no compartirlas ni subirlas a ningún repositorio. La única excepción son las llamadas a Jev de la 4.3, que envían las letras del test a OpenRouter.

### Flujo de trabajo

Cada etapa parte explorando en el notebook: probamos una idea y miramos qué resulta. Cuando una parte de esa exploración sirve más de una vez, la pasamos a una función de `src/radio_fcfm/`; al final de la etapa, un recuadro de modularización dice cuáles funciones escribir. Para saber si quedaron bien, cada etapa trae pruebas que se ejecutan desde la terminal, en la raíz del laboratorio:

```bash
uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
```

Una etapa se da por cerrada cuando sus pruebas pasan.

### Configuración del Ambiente

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Instalar antes de la sesión</strong>
  Abran una terminal en la raíz de este laboratorio y ejecuten el comando de abajo. El entorno completo pesa cerca de 1 GB, casi todo <code>mlflow</code>.

  <table>
    <thead>
      <tr><th>Paquete</th><th>Para qué lo usamos</th><th>Etapa</th></tr>
    </thead>
    <tbody>
      <tr><td><code>polars</code></td><td>Lectura y transformación de la tabla de canciones</td><td>1–4</td></tr>
      <tr><td><code>scikit-learn</code></td><td>TF-IDF, pipelines, validación cruzada, clasificadores y métricas</td><td>1–4</td></tr>
      <tr><td><code>wordcloud</code></td><td>Nube de palabras de la exploración</td><td>1</td></tr>
      <tr><td><code>mlflow</code></td><td>Registro local de experimentos</td><td>1–4</td></tr>
      <tr><td><code>typesafe-sdk</code></td><td>Cliente de Jev, un modelo de lenguaje por API</td><td>4</td></tr>
      <tr><td><code>pandas</code></td><td>Tablas de <code>tabla_runs()</code> y de <code>predecir_animo</code></td><td>1–4</td></tr>
      <tr><td><code>plotly</code></td><td>Nubes de palabras y gráfico de la matriz de confusión</td><td>1, 3</td></tr>
      <tr><td><code>jupyterlab</code></td><td>Entorno de notebooks</td><td>Todas</td></tr>
    </tbody>
  </table>
</div>

```bash
uv sync --locked --all-groups
```

Al abrir el notebook en VS Code, seleccionen como kernel el `.venv` que ese comando crea en la raíz del laboratorio.

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Descargar los datos</strong>
  Los datos no vienen en el repositorio. Descarguen <code>letras.csv</code> desde el enlace que comparte el equipo docente y péguenlo en <code>data/raw/</code>, de modo que quede <code>data/raw/letras.csv</code>. El archivo trae letras de canciones, que tienen derechos de autor: es solo para este curso, y <code>data/raw/README.md</code> explica de dónde sale cada columna.
</div>

## Contexto

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Radio fcFM y su pregunta</strong>

  **Radio fcFM** es una radio que decide su programación con datos y evidencia científica, no con corazonadas. Un análisis riguroso de su audiencia, franja por franja, mostró un patrón claro: en las mañanas la gente pide música alegre; en la hora del taco, canciones con rabia, de las que se cantan a gritos con las ventanas arriba; y en la noche, algo triste o tranquilo.

  Hoy cada canción se asigna a una franja a oído, y depende de quién esté de turno. **Gaspar Ibáñez**, el director de programación, tiene un catálogo de miles de canciones que la **Fonoteca Mapocho** le despacha solo con artista, título y letra, sin ninguna etiqueta de ánimo, y no puede pagar a alguien que las escuche todas.

  Los contrataron para su primera tarea: responder una pregunta que Gaspar quiere contestar con datos y no con intuición, **¿se puede conocer el ánimo de una canción solo a partir de su letra?** Gaspar confía en sus habilidades de automatización: si la respuesta es que sí, no hará falta ni siquiera escuchar las canciones antes de elegirlas para cada franja. Para averiguarlo, les entrega un catálogo de referencia de 2 000 canciones que oyentes ya etiquetaron.
</div>

## Objetivos

- Identificar la unidad de observación de una tabla de texto y limpiarla antes de modelar.
- Apartar un conjunto de test por artista y mirarlo una sola vez, al final.
- Construir un `Pipeline` de texto (TF-IDF y un clasificador) y evaluarlo con una validación que no filtre información.
- Detectar una fuga de datos que infla el puntaje y explicar por qué no sirve en producción.
- Reconocer el sobreajuste al variar la regularización, y comparar un árbol de decisión con un bosque.
- Evaluar un clasificador con accuracy, precisión, recall y F1, y leer sus errores en una matriz de confusión.
- Registrar los experimentos en un sistema de tracking local y usarlos para fundamentar una recomendación.
- Probar un modelo de lenguaje por API (Jev) y compararlo con el modelo propio, con las mismas métricas y el mismo test.
- Predecir el ánimo de canciones nuevas con el modelo elegido.

### Evaluación

<!-- EVALUACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8b5cf6; background:rgba(139,92,246,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📊 Distribución del puntaje</strong>

| Etapa | Contenido | Puntaje |
| --- | --- | ---: |
| 1 | Datos, test y baselines | 1,0 |
| 2 | Del texto a un pipeline sin fuga | 2,0 |
| 3 | Métricas y errores | 1,5 |
| 4 | Decisión, test, Jev y canciones propias | 1,5 |
| **Total** | | **6,0** |

  La nota se calcula como 1,0 + puntaje obtenido.
</div>

## Introducción conceptual

Antes de pasar a los datos, repasemos las ideas que usaremos durante todo el laboratorio. Es solo el concepto: cada una se retoma, con código, donde se necesita.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Aprendizaje supervisado y clasificación</strong>

  En el Lab8 agrupamos personajes sin que ninguna columna dijera cuál era la respuesta correcta, y en el Lab9 predijimos un número. En este laboratorio la respuesta es una categoría. En el **aprendizaje supervisado**, cada ejemplo de entrenamiento trae su **etiqueta**, el valor correcto que el modelo debe aprender a predecir, y el modelo ajusta sus parámetros para acercarse a ella.

  Cuando la etiqueta es una categoría de un conjunto cerrado, el problema es de **clasificación**: **binaria** si hay dos categorías (un correo es spam o no lo es) y **multiclase** si hay más (la especie de una planta). A diferencia de la regresión, donde un error tiene tamaño, una predicción de clasificación solo acierta o falla (en la 3.2 veremos que no todos los errores pesan igual). Acá clasificamos **texto**: la entrada es un texto, y un modelo necesita antes una representación numérica de él, que construiremos en la Etapa 2.

  El diagrama ordena los problemas supervisados según la etiqueta que se quiere predecir.

```mermaid
flowchart TD
    S["Aprendizaje supervisado<br/>datos con etiqueta → predecir la etiqueta"] --> R["Regresión<br/>la etiqueta es un número<br/>el error tiene tamaño"]
    S --> C["Clasificación<br/>la etiqueta es una categoría<br/>la predicción acierta o falla"]
    R --> R1["Ejemplo: el precio de una vivienda"]
    C --> C1["Binaria<br/>ej.: un correo es spam o no"]
    C --> C2["Multiclase<br/>ej.: la especie de una planta"]
```

</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Entrenamiento, validación y test</strong>

  Un modelo supervisado se ajusta con ejemplos y después se usa con datos que no ha visto, así que los datos se separan antes de ajustar. El **entrenamiento** (*train*) es lo que el modelo ve para ajustar sus parámetros. La **validación** son datos que el modelo no ve al ajustarse, y con ellos se compara y se elige entre modelos o entre valores de un parámetro. El número que importa se calcula sobre datos que el modelo no usó: un modelo puede memorizar su entrenamiento y fallar con datos nuevos. La **validación cruzada**, que usamos en el Lab9, repite esa separación varias veces rotando qué parte queda afuera, de modo que cada ejemplo se predice con un modelo que no lo vio.

  Todo lo que se decide mirando la validación la gasta un poco: el mejor de varios candidatos queda optimista. Por eso se reserva un conjunto de **test** (*holdout*): datos que se apartan al comienzo, que ninguna decisión usa y que se miran **una sola vez por modelo finalista**, con cada modelo ya fijo y sin ajustar nada después. Esa cifra es la que se le promete a quien usará el modelo.

  El primer diagrama muestra las particiones: el test se aparta al comienzo y solo se toca al final, mientras que el resto de los datos se reparte entre train (ajustar) y validation (comparar).

```mermaid
%%{init: {"themeVariables": {"fontSize": "12px"}, "flowchart": {"nodeSpacing": 10, "rankSpacing": 14, "padding": 4}}}%%
flowchart LR
    D["Datos<br/>100 %"] --> DEV["Train y validation<br/>p. ej. 80 %"]
    D --> T["Test<br/>p. ej. 20 %"]
    DEV --> TR["Entrenamiento"]
    DEV --> V["Validación"]
    T --> E["Se mira una vez<br/>por modelo finalista"]
```

  El segundo muestra la validación cruzada con cinco particiones. Los datos que no son test se dividen en cinco partes y, en cada vuelta, una parte distinta hace de validación (naranja) mientras las otras cuatro entrenan (azul). El puntaje de validación cruzada es el promedio de los cinco puntajes, y cada ejemplo se predijo una vez con un modelo que no lo vio.

```mermaid
%%{init: {"themeVariables": {"fontSize": "12px"}, "flowchart": {"nodeSpacing": 10, "rankSpacing": 14, "padding": 4}}}%%
flowchart TB
    subgraph V1["Vuelta 1"]
        direction LR
        A1["Validación"] ~~~ A2["Entrenamiento"] ~~~ A3["Entrenamiento"] ~~~ A4["Entrenamiento"] ~~~ A5["Entrenamiento"]
    end
    subgraph V2["Vuelta 2"]
        direction LR
        B1["Entrenamiento"] ~~~ B2["Validación"] ~~~ B3["Entrenamiento"] ~~~ B4["Entrenamiento"] ~~~ B5["Entrenamiento"]
    end
    subgraph V3["Vuelta 3"]
        direction LR
        C1["Entrenamiento"] ~~~ C2["Entrenamiento"] ~~~ C3["Validación"] ~~~ C4["Entrenamiento"] ~~~ C5["Entrenamiento"]
    end
    subgraph V4["Vuelta 4"]
        direction LR
        D1["Entrenamiento"] ~~~ D2["Entrenamiento"] ~~~ D3["Entrenamiento"] ~~~ D4["Validación"] ~~~ D5["Entrenamiento"]
    end
    subgraph V5["Vuelta 5"]
        direction LR
        E1["Entrenamiento"] ~~~ E2["Entrenamiento"] ~~~ E3["Entrenamiento"] ~~~ E4["Entrenamiento"] ~~~ E5["Validación"]
    end
    V1 ~~~ V2 ~~~ V3 ~~~ V4 ~~~ V5
    classDef val fill:#f0883e,stroke:#b35900,color:#000
    classDef ent fill:#58a6ff,stroke:#1f6feb,color:#000
    class A1,B2,C3,D4,E5 val
    class A2,A3,A4,A5,B1,B3,B4,B5,C1,C2,C4,C5,D1,D2,D3,D5,E1,E2,E3,E4 ent
```

</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Métricas de clasificación</strong>

  Una **métrica** es un número que resume qué tan bien predice un modelo. Para entender las de clasificación, pensemos en un filtro de correo y en la clase `spam`. Cada predicción cae en uno de tres casos: es un **verdadero positivo** (VP) si el filtro dijo `spam` y el correo lo era; un **falso positivo** (FP), una falsa alarma, si dijo `spam` y era un correo legítimo; o un **falso negativo** (FN), un caso que se escapa, si el correo era `spam` y el filtro lo dejó pasar.

  **Accuracy.** Es la fracción de predicciones correctas sobre el total: $\text{accuracy} = \frac{\text{predicciones correctas}}{\text{total de ejemplos}}$. Es la métrica más simple y entrega un solo número para todas las clases juntas, así que no dice cuál de ellas falla. Además engaña cuando las clases no están parejas: un modelo que siempre responde la clase más común acierta mucho sin reconocer nada más.

  **Precisión.** Se calcula para una clase: de todo lo que el modelo predijo como esa clase, qué fracción lo era de verdad, $P = \frac{VP}{VP + FP}$. Una precisión baja significa muchas falsas alarmas. Para el filtro, es la fracción de los correos marcados como `spam` que de verdad lo eran: si es baja, correos legítimos terminan en la carpeta de spam.

  **Recall.** También se calcula para una clase: de todo lo que era esa clase, qué fracción encontró el modelo, $R = \frac{VP}{VP + FN}$. Un recall bajo significa casos reales que se escapan. Para el filtro, es la fracción de los correos spam que logra detectar. Precisión y recall miden cosas distintas y suelen competir: un filtro que marca casi todo como `spam` tiene un recall altísimo y una precisión baja.

  **F1.** Resume la precisión y el recall de una clase en un solo número con su media armónica, $F_1 = \frac{2PR}{P + R}$. La media armónica es baja si cualquiera de las dos lo es, así que no deja compensar una con la otra: con $P = 1$ y $R = 0{,}1$, el $F_1$ es cerca de $0{,}18$, y no el $0{,}55$ que daría el promedio simple.

  **F1 macro.** Cuando hay varias clases, el F1 macro promedia los F1 de todas con el mismo peso, sin importar cuántos ejemplos tiene cada una. Así, una clase que el modelo casi nunca acierta lo arrastra hacia abajo aunque sea chica. Su alternativa, el **F1 ponderado**, promedia ponderando por el tamaño de cada clase, y por eso se parece más al accuracy.

  Cuál métrica conviene depende de la decisión que se tomará con el resultado, y lo retomamos en la Etapa 3.
</div>

## El dataset

Trabajaremos con **MoodyLyrics4Q**, un dataset de **2 000 canciones** con una etiqueta de ánimo: `happy`, `angry`, `sad` o `relaxed`, 500 de cada una. Las etiquetas salen de los tags que usuarios de Last.fm pusieron a cada canción, y corresponden a los cuatro cuadrantes del modelo circumplejo del afecto de Russell.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Modelo circumplejo del afecto</strong>

  El **modelo circumplejo del afecto**, propuesto por el psicólogo James A. Russell en 1980, plantea que las emociones no son categorías aisladas, sino puntos de un espacio de dos dimensiones: la **valencia** (eje horizontal), que va del displacer al placer, y la **activación** (eje vertical), que va de la calma a la alta energía o alerta. Todo estado de ánimo es una combinación de las dos, y los cuatro cuadrantes del plano son los cuatro ánimos de este laboratorio.

  <div style="text-align: center; margin: 16px 0;">
    <img src="static/russell-model.png" alt="Modelo circumplejo del afecto de Russell: un plano con un eje horizontal de valencia, de desagradable a agradable, y un eje vertical de activación, de desactivación a activación. Los cuatro cuadrantes son angry (arriba a la izquierda), happy (arriba a la derecha), sad (abajo a la izquierda) y relaxed (abajo a la derecha), cada uno con emociones como tense, excited, depressed y calm." style="width: 500px; max-width: 100%; height: auto;">
  </div>

  *Fuente de la figura: Candra, H., Yuwono, M., Handojoseno, A., Chai, R., Su, S. y Nguyen, H. (2015). Recognizing emotions from EEG subbands using wavelet analysis. doi: 10.1109/EMBC.2015.7319766.*

</div>

`data/raw/letras.csv` trae una fila por canción:

| Columna | Qué es | Rol |
| --- | --- | --- |
| `Index` | Identificador de la canción en MoodyLyrics4Q (`ML1`, `ML2`, …) | Clave |
| `Artist` | Artista | Dato descriptivo |
| `Title` | Título | Dato descriptivo |
| `Lyrics` | Letra completa en texto plano; vacía si no se encontró | **Entrada** |
| `Mood` | `happy`, `angry`, `sad` o `relaxed` | **Etiqueta** |
| `Idioma` | Código ISO 639-3 del idioma que detectó el equipo docente (`eng` es inglés) | Filtro |

La etiqueta describe la canción completa tal como la percibieron sus oyentes —letra, sonido, género—, no solo lo que dice la letra. Tengan presente esa diferencia: el laboratorio mide cuánto de ese ánimo se puede recuperar leyendo.

### Rutas del laboratorio

In [1]:
from pathlib import Path
import sys

RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

RAW_DIR = Path("../data/raw")

### MLflow

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Recordatorio — MLflow</strong>

  **MLflow** registra los experimentos de modelado. Cada intento de entrenar y evaluar un modelo se guarda como un **run**, con sus **parámetros** (qué se probó), sus **métricas** (cuánto rindió), sus **tags** (etiquetas libres para clasificarlo) y sus **artefactos** (archivos, como el modelo ajustado). Los runs se agrupan en un **experimento**.

  Sirve para comparar intentos sin depender de la memoria ni de notas sueltas, y para recuperar después, tal cual, el modelo que se eligió. En su forma local no necesita servidor: los runs quedan en una base SQLite y los artefactos en una carpeta.
</div>

Esta celda deja listo MLflow: la base `mlflow.db` y los artefactos (`mlruns/`) quedan en la carpeta del laboratorio, y todos los runs van al experimento `radio-fcfm`. También baja el volumen de los mensajes de MLflow para que no tapen las salidas del notebook.

In [2]:
import logging
import os
import warnings

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")

import mlflow

from src.radio_fcfm.contratos import NOMBRE_EXPERIMENTO

logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.set_tracking_uri(f"sqlite:///{RUTA_LAB}/mlflow.db")
if mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO) is None:
    mlflow.create_experiment(
        NOMBRE_EXPERIMENTO, artifact_location=(RUTA_LAB / "mlruns").as_uri()
    )
experimento = mlflow.set_experiment(NOMBRE_EXPERIMENTO)
print(f"Experimento activo: {experimento.name}")

Experimento activo: radio-fcfm


---
## Etapa 1 — Datos, test y baselines (1,0 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Un catálogo de referencia con dos advertencias</strong>

  Gaspar entrega el catálogo de referencia con dos advertencias. La primera: de las 2 000 canciones, no todas traen su letra. La segunda viene del propio problema: un sistema que va a clasificar un catálogo real hay que medirlo con artistas que el equipo nunca haya mirado, porque un modelo puede aprender a reconocer a un artista y no el ánimo de sus canciones. Por eso una parte de los artistas se guarda sin tocar hasta que se elija el modelo.
</div>

### 1.1 — Cargar y contar (0,2 puntos)

`data/raw/letras.csv` trae una fila por canción. Carguémoslo y miremos su forma antes de tocar nada.

In [3]:
import polars as pl

ruta_letras = RAW_DIR / "letras.csv"
if not ruta_letras.exists():
    raise FileNotFoundError(
        f"No existe {ruta_letras.resolve()}. Descarguen letras.csv desde el "
        "enlace del equipo docente y péguenlo en data/raw/ "
        "(ver data/raw/README.md)."
    )
canciones = pl.read_csv(ruta_letras)
canciones.head()

Index,Artist,Title,Mood,Lyrics,Idioma
str,str,str,str,str,str
"""ML1""","""George Michael""","""I Want Your Sex""","""happy""","""There's things that you guess …","""eng"""
"""ML2""","""Rob Zombie""","""Pussy Liquor""","""angry""","""Earl had a baby Baby was her …","""eng"""
"""ML5""","""Ludacris""","""Get Back""","""angry""","""Heads up! Heads up! Here's an…","""eng"""
"""ML6""","""Hole""","""Violet""","""angry""","""And the sky was made of amethy…","""eng"""
"""ML7""","""Manic Street Preachers""","""Ifwhiteamericatoldthetruthforo…","""angry""","""Next Thursday you're invited t…","""eng"""


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de contar</strong>
  1. ¿Qué identifica a una fila de `canciones`: la canción, el artista o la letra? ¿Qué columna sirve de clave, y por qué `Title` no sirve?
  2. El catálogo tiene 2 000 canciones, 500 de cada ánimo. Si algunas no traen letra, ¿qué esperan que pase con ese balance?
  3. Algunas canciones se parecen entre sí por algo más que su ánimo, por ejemplo porque las canta la misma persona. ¿Qué columna permite agrupar esas canciones, y por qué podría importar al evaluar un modelo?
</div>

`Escriban sus respuestas aquí:`
1. Cada fila es una canción. La clave es `Index` (`ML1`, `ML2`, …): hay 2 000 filas y 2 000 `Index` distintos. `Title` no sirve porque se repite: hay solo 1 978 títulos distintos, porque canciones distintas, de artistas distintos, pueden llamarse igual.
2. Si las letras faltan sin relación con el ánimo, el balance se mantiene casi parejo pero deja de ser exacto, porque cada ánimo pierde un número distinto de canciones. Así pasó: quedaron 488 `angry`, 482 `sad`, 479 `happy` y 476 `relaxed`.
3. `Artist`. Las canciones de un mismo artista comparten estilo, vocabulario y, muchas veces, ánimo. Si un artista queda a la vez en entrenamiento y en validación, el modelo puede reconocer al artista en lugar del ánimo y el puntaje sale optimista; por eso hay que agrupar por artista al separar los datos.

Contemos lo que las respuestas anteriores dan por sabido: cuántas filas hay y cuántos `Index` y `Title` distintos.

In [4]:
print(f"Filas: {canciones.height}")
print(f"Index distintos: {canciones['Index'].n_unique()}")
print(f"Title distintos: {canciones['Title'].n_unique()}")

Filas: 2000
Index distintos: 2000
Title distintos: 1978


Polars lee un campo vacío como **nulo**, no como texto vacío: las canciones sin letra aparecen con `Lyrics` nulo.

Construyan `con_letra`, las canciones cuya `Lyrics` no es nula ni está formada solo por espacios. Deben quedar 1925 filas.

(Hint: `filter`, `str.strip_chars`, `str.len_chars`)

In [5]:
# Nos quedamos con las canciones que sí traen letra: `Lyrics` no puede ser
# nula (campo vacío en el CSV) ni estar formada solo por espacios. Para lo
# segundo, se quitan los espacios de los extremos y se exige al menos un
# carácter.
con_letra = canciones.filter(
    pl.col("Lyrics").is_not_null()
    & (pl.col("Lyrics").str.strip_chars().str.len_chars() > 0)
)
print(f"Canciones con letra: {con_letra.height}")

Canciones con letra: 1925


Veamos cuántas canciones quedaron en cada ánimo.

In [6]:
print(f"Canciones con letra: {con_letra.height} de {canciones.height}")
con_letra["Mood"].value_counts().sort("Mood")

Canciones con letra: 1925 de 2000


Mood,count
str,u32
"""angry""",488
"""happy""",479
"""relaxed""",476
"""sad""",482


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Las canciones sin letra</strong>
  Faltan las letras de unas cuantas canciones. ¿Por qué no tiene sentido imputarlas, como se haría con una columna numérica a la que le faltan valores?
</div>

`Escriban su respuesta aquí:`
1. Lo que falta es la entrada misma que el modelo debe leer, no un valor numérico que se pueda aproximar con un promedio. Cualquier relleno (texto vacío, una letra "típica" o la de otra canción) no tiene relación con la canción real, así que le enseñaría al modelo asociaciones falsas; y si el relleno dependiera del ánimo, sería fuga. Además son pocas (75 de 2 000, un 3,75 %) y repartidas parejo entre los ánimos, así que excluirlas casi no cambia el balance.

### 1.2 — Limpiar las letras (0,2 puntos)

Antes de limpiar el texto hay que decidir qué canciones entran. El modelo va a trabajar con las palabras de las letras, y las palabras de otro idioma no se parecen a las del inglés. `letras.csv` trae la columna `Idioma`, con el código ISO 639-3 del idioma que el equipo docente detectó para cada letra con el clasificador de idioma de nltk (`eng` es inglés). Veamos cuántas canciones hay de cada idioma.

In [7]:
con_letra.group_by("Idioma").len().sort("len", descending=True)

Idioma,len
str,u32
"""eng""",1855
"""deu""",14
"""por""",14
"""fra""",13
"""nds""",5
…,…
"""tur""",1
"""nob""",1
"""swh""",1


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Las canciones que no están en inglés</strong>
  1. ¿Qué harían con las canciones que no están en inglés: dejarlas, traducirlas o excluirlas? ¿Qué le pasaría al modelo si se quedaran?
  2. El idioma lo decidió un clasificador automático. ¿Qué riesgo tiene excluir según su decisión?
</div>

`Escriban sus respuestas aquí:``Escriban sus respuestas aquí:`
1. Excluirlas, ya que, son 70 de 1 925 (cerca de 3,6 %) repartidas en 20 idiomas, con muy pocas por idioma. Sus palabras no comparten vocabulario con el inglés: el TF-IDF las representaría con palabras que casi no aparecen en otras canciones (o que `min_df=2` descarta), y las palabras vacías que se quitan son las del inglés. El modelo aprendería ruido, o asociaría un idioma con un ánimo. Traducirlas agregaría los errores de la traducción y un paso extra que habría que repetir en producción; con tan pocas no se justifica.
2. El clasificador de idioma se equivoca (el README advierte que confunde idiomas y falla con letras cortas). Puede excluir canciones que sí están en inglés (marcadas `und` o con otro idioma por error) y dejar alguna que no lo está. Si sus errores se concentran en cierto tipo de canción (letras cortas, con muchos "la la la"), la exclusión podría sesgar el catálogo hacia algún ánimo. Conviene revisar a mano una muestra de las excluidas.

Las letras vienen de una base colaborativa y algunas traen marcas de estructura que no son parte de lo que se canta: entre corchetes, como `[Verse 1]` o `[Chorus]`, y algunas entre paréntesis, como `(Chorus)`. Si se dejan, el modelo las trata como palabras de la letra. Veamos cuántas canciones las traen y cómo se ven.

In [8]:
con_corchetes = con_letra.filter(
    pl.col("Lyrics").str.contains("[", literal=True)
)
print(
    f"Letras con algún corchete: {con_corchetes.height} de {con_letra.height}"
)
print(con_corchetes["Lyrics"][0][:160])

con_coro = con_letra.filter(pl.col("Lyrics").str.contains("(Chorus)", literal=True))
print(f"Letras con (Chorus) entre paréntesis: {con_coro.height}")

Letras con algún corchete: 29 de 1925
That's how it starts
We go back to your house
You check the charts
And start to figure it out

And if it's crowded, all the better
Because we know we're g
Letras con (Chorus) entre paréntesis: 8


Construyan `letras_limpias`, una copia de `con_letra` que conserve solo las canciones en inglés (`Idioma` igual a `"eng"`) y tenga una columna nueva, `Letra`, que traiga la letra sin las marcas de sección y con los espacios normalizados: un solo espacio entre palabras y ningún espacio ni salto de línea al inicio o al final. Las marcas son las que van entre corchetes (`[Verse 1]`) y las que van entre paréntesis y empiezan con `Chorus`, `Verse`, `Bridge`, `Intro`, `Outro` o `Instrumental`, sin importar las mayúsculas (`(Chorus)`, `(Instrumental break)`). Reemplacen cada marca por un espacio. Dejen intactos los demás paréntesis, porque suelen ser coros (`(oh oh)`) y sí se cantan.

(Hint: `filter`, `with_columns`, `str.replace_all`, `str.strip_chars`; en una expresión regular, `(?i)` ignora las mayúsculas)

In [9]:
# Marca entre corchetes, como [Verse 1], [x2] o una parte hablada que ocupa
# varias líneas. `[^\[\]]*` no deja que la marca cruce otro corchete: así el
# corchete suelto de ML271 ("p[lace") no se come el texto hasta el "]" de la
# marca siguiente.
MARCA_CORCHETES = r"\[[^\[\]]*\]"
# Marca de sección entre paréntesis: (Chorus), (Instrumental break), etc.
# `(?i)` ignora mayúsculas y `\b` exige que la palabra termine ahí, para no
# tocar paréntesis como "(Introducing love)".
MARCA_PARENTESIS = (
    r"(?i)\((?:chorus|verse|bridge|intro|outro|instrumental)\b[^()\n]*\)"
)

letras_limpias = (
    # Solo las canciones en inglés, según la columna Idioma.
    con_letra.filter(pl.col("Idioma") == "eng")
    .with_columns(
        pl.col("Lyrics")
        # Cada marca se reemplaza por un espacio, para no pegar palabras.
        .str.replace_all(MARCA_CORCHETES, " ")
        .str.replace_all(MARCA_PARENTESIS, " ")
        # Cualquier secuencia de espacios, tabs o saltos de línea -> un espacio.
        .str.replace_all(r"\s+", " ")
        # Sin espacios al inicio ni al final.
        .str.strip_chars()
        .alias("Letra")
    )
)
print(f"Canciones en inglés, limpias: {letras_limpias.height}")

Canciones en inglés, limpias: 1855


<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  <code>letras_limpias</code> debe tener 1855 filas: las 1925 con letra menos las 70 que no están en inglés. La celda de abajo debe mostrar <strong>una sola</strong> letra con un corchete: la de <code>ML271</code>, que trae un corchete suelto dentro de una palabra (<code>p[lace</code>), un error de tipeo de la fuente. No es una marca de estructura y la limpieza no debe tocarla. La línea de abajo debe mostrar <strong>0</strong> letras con una marca de sección entre paréntesis.
</div>

In [10]:
sobrevivientes = letras_limpias.filter(
    pl.col("Letra").str.contains("[", literal=True)
)
print(f"Letras que aún traen un corchete: {sobrevivientes.height}")
marcas_restantes = letras_limpias.filter(
    pl.col("Letra").str.contains(
        r"(?i)\((?:chorus|verse|bridge|intro|outro|instrumental)\b"
    )
)
print(
    "Letras que aún traen una marca de sección entre paréntesis: "
    f"{marcas_restantes.height}"
)
sobrevivientes.select("Index", "Artist", "Title")

Letras que aún traen un corchete: 1
Letras que aún traen una marca de sección entre paréntesis: 0


Index,Artist,Title
str,str,str
"""ML271""","""Delays""","""Touch Down"""


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ ¿Afuera o adentro del pipeline?</strong>
  Acabamos de limpiar las 1855 letras juntas, antes de cualquier validación. ¿Es una fuga de datos?
</div>

`Escriban su respuesta aquí:`
1. No es fuga, la limpieza es una regla fija que se aplica a cada letra por separado, no aprende nada de las otras letras ni mira la etiqueta, así que una letra queda igual se limpie sola o junto con todo el corpus. La fuga aparece cuando un paso **aprende** de los datos (como el vocabulario y los pesos IDF del TF-IDF, o una selección de palabras) y ese aprendizaje usa las letras de validación. Por eso el TF-IDF va dentro del pipeline y la limpieza puede ir afuera; de hecho, `predecir_animo` aplica la misma limpieza a cada letra nueva.

### 1.3 — Apartar el test (0,2 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 El test se aparta por grupo</strong>

  El conjunto de test de la introducción se aparta al comienzo, antes de explorar el catálogo (1.4) y de entrenar nada. Un call center que clasifica mensajes de clientes debe apartarlo por **grupo**, no por mensaje: si un cliente tiene mensajes a ambos lados, el test mide si el modelo reconoce a ese cliente y no si funciona con uno nuevo. Además debe conservar todas las categorías, es decir, estar **estratificado**. De ahí en adelante, «los datos» son todo lo que no es test: el entrenamiento (*train*) y la validación (*validation*), que la validación cruzada reparte de nuevo en cada partición.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de apartar el test</strong>
  Si el test se armara sorteando canciones y no artistas, ¿qué esperan que le pase a su puntaje frente al que tendrá con un artista nuevo? ¿Por qué?
</div>

`Escriban su respuesta aquí:`
1. El puntaje saldría más alto, es decir, optimista. Canciones de un mismo artista quedarían a ambos lados, el modelo aprendería el vocabulario y el estilo de ese artista, y en el test lo reconocería en vez de reconocer el ánimo (en la 1.4 se ve que muchos artistas tienen todas sus canciones en un mismo ánimo). Con un artista nuevo esa ventaja desaparece, así que el test por canciones prometería más de lo que el modelo rendirá con el catálogo real de la Fonoteca.

Preparemos los arreglos de las 1855 canciones en inglés: la letra limpia, el ánimo y el artista, en el mismo orden que `letras_limpias`.

In [11]:
textos_todos = letras_limpias["Letra"].to_numpy()
estados_todos = letras_limpias["Mood"].to_numpy()
artistas_todos = letras_limpias["Artist"].to_numpy()

Construyan `train_val` y `test`, los arreglos con las posiciones de las canciones de cada conjunto (`train_val` junta el train y la validation). Usen un `StratifiedGroupKFold` de 5 particiones, con `shuffle=True` y `random_state=0`, que estratifique por `estados_todos` y agrupe por `artistas_todos`: su primera partición de validación es el `test` (un 20 % de los artistas) y el resto es `train_val`.

(Hint: `StratifiedGroupKFold`, `split`)

In [12]:
from sklearn.model_selection import StratifiedGroupKFold

# Cinco particiones que agrupan por artista (ningún artista queda en dos
# particiones) y estratifican por ánimo (los cuatro ánimos en cada una).
# `shuffle=True` con `random_state=0` hace la partición reproducible.
particion_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

# La primera partición: su parte de validación (~20 % de los artistas) es el
# test; el resto (train y validation) queda en `train_val`.
train_val, test = next(
    particion_test.split(textos_todos, estados_todos, groups=artistas_todos)
)
print(f"train_val: {len(train_val)} canciones | test: {len(test)} canciones")

train_val: 1483 canciones | test: 372 canciones


De aquí en adelante, `textos`, `estados` y `artistas` son **solo train y validation**, sin el test. El test queda guardado en `textos_test` y `estados_test`, y no se vuelve a nombrar hasta la Etapa 4.

In [13]:
textos = textos_todos[train_val]
estados = estados_todos[train_val]
artistas = artistas_todos[train_val]
letras_train_val = letras_limpias[train_val]

textos_test = textos_todos[test]
estados_test = estados_todos[test]
artistas_test = artistas_todos[test]

print(f"Train y validation: {len(textos)} canciones de {len(set(artistas))} artistas")
n_artistas_test = len(set(artistas_test))
print(
    f"Test: {len(textos_test)} canciones de {n_artistas_test} artistas"
)
print(f"Artistas en común: {len(set(artistas) & set(artistas_test))}")
pl.Series("Mood", estados_test).value_counts().sort("Mood")

Train y validation: 1483 canciones de 953 artistas
Test: 372 canciones de 240 artistas
Artistas en común: 0


Mood,count
str,u32
"""angry""",97
"""happy""",92
"""relaxed""",91
"""sad""",92


<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  <code>train_val</code> debe tener 1483 canciones y <code>test</code>, 372. Los artistas en común deben ser <strong>0</strong>, y cada ánimo debe aparecer en el test con entre 91 y 97 canciones.
</div>

### 1.4 — Explorar el catálogo (0,2 puntos)

Empecemos por los artistas. Miremos quiénes tienen más canciones entre los datos y cuántos ánimos distintos recorren.

In [14]:
por_artista = letras_train_val.group_by("Artist").agg(
    pl.len().alias("canciones"),
    pl.col("Mood").n_unique().alias("animos"),
    pl.col("Mood").mode().first().alias("animo_frecuente"),
)
varias = por_artista.filter(pl.col("canciones") >= 3)
solo_uno = varias.filter(pl.col("animos") == 1).height
print(f"Artistas con 3 o más canciones: {varias.height}")
print(f"  con todas sus canciones en un solo ánimo: {solo_uno}")
por_artista.sort(["canciones", "Artist"], descending=[True, False]).head(10)

Artistas con 3 o más canciones: 99
  con todas sus canciones en un solo ánimo: 59


Artist,canciones,animos,animo_frecuente
str,u32,u32,str
"""Megadeth""",26,1,"""angry"""
"""Rage Against the Machine""",22,1,"""angry"""
"""Nine Inch Nails""",12,2,"""angry"""
"""Ice Cube""",10,1,"""angry"""
"""The Cure""",10,3,"""sad"""
"""Britney Spears""",8,2,"""happy"""
"""Enya""",8,1,"""relaxed"""
"""Jack Johnson""",8,1,"""relaxed"""
"""Marilyn Manson""",8,2,"""angry"""


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Los artistas del catálogo</strong>
  1. Entre los diez artistas con más canciones, ¿cuántos tienen todas las suyas en un solo ánimo? ¿Cuál es el ánimo más frecuente entre ellos?
  2. De los artistas con 3 o más canciones, ¿cuántos tienen todas en un solo ánimo? ¿Qué le dice eso a un modelo que podría reconocer al artista, y qué implica para cómo hay que evaluarlo?
</div>

`Escriban sus respuestas aquí:`
1. 6 de los 10 tienen todas sus canciones en un solo ánimo. El ánimo más frecuente entre ellos es `angry`, que es el principal en 5 de los 10.
2. 59 de los 99 artistas con 3 o más canciones tienen todas sus canciones en un solo ánimo. Para un modelo, el artista (su vocabulario, sus temas, sus nombres propios) funciona casi como una etiqueta, puede aprender "suena a Megadeth → `angry`" sin entender el ánimo. Si se evalúa con canciones del mismo artista en entrenamiento y validación, ese atajo infla el puntaje; hay que evaluar agrupando por artista, como en el test de la 1.3 y en la validación agrupada.

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Las celdas de arriba deben mostrar 99 artistas con 3 o más canciones, 59 de ellos con un solo ánimo, y una tabla que abre con <code>Megadeth</code> (26 canciones) y <code>Rage Against the Machine</code> (22).
</div>

Ahora miremos las palabras más recurrentes de cada ánimo a través de una nube de palabras.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Nube de palabras</strong>

  Una **nube de palabras** dibuja las palabras más frecuentes de un conjunto de textos, con un tamaño proporcional a su frecuencia. Sirve para mirar los datos antes de modelar, pero solo muestra qué palabras abundan, no cuáles distinguen una categoría de otra.
</div>

Los conectores y otras palabras de uso general no aportan información para deducir el ánimo de una canción, y con su tamaño dominarían la nube. Por eso removeremos las *stopwords*: una lista de palabras que se eliminan de los datos con que se esté trabajando.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Palabras vacías (stopwords)</strong>

  Las **palabras vacías** (*stopwords*) son las que aparecen en casi cualquier texto, sea cual sea su tema: conectores, artículos, preposiciones. Por eso no sirven para distinguir una categoría de otra. Cada idioma tiene su lista fija, y las palabras que contiene se eliminan antes de contar. Acá usaremos la lista del inglés que trae scikit-learn.
</div>

Para dibujar las nubes necesitamos antes saber cuántas veces aparece cada palabra en las letras de cada ánimo. Construyan `frecuencias_por_estado`: un diccionario que asigne a cada ánimo de `ESTADOS` (en `src/radio_fcfm/contratos.py`) otro diccionario de palabra a número de veces que aparece en las letras de ese ánimo, descartando las *stopwords* del inglés. Deben quedar cuatro ánimos, cada uno con miles de palabras distintas.

(Hint: `CountVectorizer`, `stop_words`, `get_feature_names_out`)

In [15]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

from src.radio_fcfm.contratos import ESTADOS

frecuencias_por_estado = {}
# Un `for` sobre los cuatro ánimos (no sobre las canciones): se repite el
# mismo conteo para cada uno.
for estado in ESTADOS:
    # Letras de train y validation de este ánimo (el test no se mira).
    letras_estado = letras_train_val.filter(pl.col("Mood") == estado)[
        "Letra"
    ].to_list()
    # CountVectorizer cuenta cuántas veces aparece cada palabra en cada letra,
    # descartando las palabras vacías del inglés.
    contador = CountVectorizer(stop_words="english")
    conteos = contador.fit_transform(letras_estado)
    # Sumando por columna se obtiene el total de cada palabra en el ánimo.
    totales = np.asarray(conteos.sum(axis=0)).ravel()
    frecuencias_por_estado[estado] = dict(
        zip(contador.get_feature_names_out(), totales.tolist(), strict=True)
    )
    print(f"{estado:8s} {len(frecuencias_por_estado[estado])} palabras distintas")

happy    6362 palabras distintas
relaxed  4781 palabras distintas
sad      4849 palabras distintas
angry    8185 palabras distintas


Dibujemos una nube por ánimo, cada una en su propia figura.

In [16]:
import plotly.express as px
from wordcloud import WordCloud

from src.radio_fcfm.contratos import ESTADOS

for estado in ESTADOS:
    nube = WordCloud(
        width=800, height=450, background_color="white", random_state=0
    )
    nube.generate_from_frequencies(frecuencias_por_estado[estado])
    figura = px.imshow(nube.to_array(), title=f"Nube de palabras: {estado}")
    figura.update_xaxes(visible=False)
    figura.update_yaxes(visible=False)
    figura.show()

Para comparar las nubes con más precisión, listemos las quince palabras más frecuentes de cada ánimo.

In [17]:
from collections import Counter

for estado in ESTADOS:
    frecuentes = Counter(frecuencias_por_estado[estado]).most_common(15)
    print(f"{estado:8s} {[palabra for palabra, _ in frecuentes]}")

happy    ['oh', 'like', 'yeah', 'just', 'don', 'love', 'know', 'la', 'got', 'come', 'baby', 'let', 'want', 'na', 'll']
relaxed  ['love', 'know', 'don', 'just', 'like', 'oh', 'll', 've', 'away', 'baby', 'time', 'come', 'let', 'way', 'got']
sad      ['don', 'love', 'just', 'oh', 'like', 'know', 'll', 've', 'time', 'away', 'say', 'feel', 'make', 'life', 'way']
angry    ['don', 'just', 'like', 'got', 'll', 'know', 'gonna', 'fuck', 'yeah', 'time', 'cause', 'world', 'come', 'man', 've']


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Qué muestran las nubes</strong>
  1. ¿Qué palabras aparecen grandes en las cuatro nubes? ¿Sirven para distinguir un ánimo de otro?
  2. Aparecen fragmentos como `don`, `ll` y `ve`. ¿De dónde salen, y por qué no los descartó la lista de palabras vacías?
  3. ¿Qué palabras parecen propias de un ánimo? ¿Alcanza con contar cuántas veces aparece cada palabra para saber cuáles distinguen un ánimo de otro?
</div>

`Escriban sus respuestas aquí:`
1. Las mismas palabras aparecen grandes en las cuatro nubes: `don`, `just`, `like`, `know`, `oh`, `love`, `ll`, `time`, `come`, entre otras. Son frecuentes en cualquier canción, sin importar su ánimo, así que no sirven para distinguir uno de otro.
2. Salen de las contracciones del inglés: *don't* → `don` + `t`, *I'll* → `ll`, *I've* → `ve`. El tokenizador de scikit-learn corta en el apóstrofo y descarta los fragmentos de una sola letra (`t`, `s`), pero `don`, `ll` y `ve` tienen dos o más letras y no están en la lista de palabras vacías, que trae las palabras completas.
3. Algunas parecen propias de un ánimo: `fuck`, `world`, `man` y `gonna` en `angry`; `la`, `na` y `baby` en `happy`; `life`, `feel` y `away` en `sad`. Pero contar no alcanza: la frecuencia absoluta mezcla las palabras comunes a todos con las propias, y una palabra puede ser frecuente en un ánimo solo porque sus letras son más largas (`happy` y `angry` tienen letras más largas). Para saber cuáles distinguen hay que comparar entre ánimos (frecuencias relativas, TF-IDF o los pesos de un modelo, como en la 3.4).

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  <code>frecuencias_por_estado</code> debe tener cuatro ánimos, con entre unas 4 800 y 8 200 palabras distintas cada uno (la celda de la solución imprime los conteos). La celda de las nubes debe dibujar cuatro figuras, una por ánimo, con título <code>happy</code>, <code>relaxed</code>, <code>sad</code> y <code>angry</code>, sin palabras como <em>the</em> o <em>and</em>.
</div>

### 1.5 — Baselines y primer registro (0,2 puntos)

Antes de ajustar un modelo con las letras, fijemos contra qué se va a comparar.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Baseline</strong>

  Un **baseline** es un modelo simple a propósito, que sirve de referencia: marca el puntaje que cualquier modelo serio tiene que superar. Si un modelo de letras no le gana a un baseline, no está aprendiendo nada útil, y si le gana por poco conviene preguntarse si usa una señal más simple que el significado de la letra. Un baseline se mide igual que los demás modelos: con los mismos datos, la misma validación y la misma métrica, y sin el test.
</div>

En este laboratorio usaremos dos baselines:

| Baseline | Qué entrada usa | Qué predice | Qué mide |
| --- | --- | --- | --- |
| **Clase mayoritaria** | Ninguna: ignora la letra | Siempre el ánimo más frecuente | Cuánto se acierta sin saber nada de la canción |
| **Largo de la letra** | $\log(1 + n)$, con $n$ el número de palabras de la letra | El ánimo que una regresión logística asigna a partir de ese único número | Cuánto se acierta sin leer, solo contando palabras |

La regresión logística es un clasificador lineal que veremos en la Etapa 2; acá recibe un solo número por canción. El logaritmo evita que las letras muy largas dominen la escala.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 DummyClassifier</strong>

  `DummyClassifier` es la clase de scikit-learn con que se construye un baseline: un clasificador que **ignora las variables de entrada** y predice solo a partir de las etiquetas con que se ajustó. Su parámetro `strategy` decide cómo. Con `"most_frequent"`, que es el que usaremos, predice siempre la clase más frecuente. Con `"stratified"` sortea la clase respetando las proporciones del conjunto, y con `"uniform"` la sortea con igual probabilidad. Como tiene los mismos métodos `fit` y `predict` que cualquier otro estimador, se evalúa con la misma función y se compara con los modelos reales en igualdad de condiciones.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Accuracy y F1 macro en un ejemplo</strong>

  Veamos cómo se comportan estas métricas con un call center que deriva cada mensaje a <em>reclamo</em>, <em>consulta</em> o <em>felicitación</em>. Si los mensajes se reparten parejo entre las tres categorías y un clasificador responde siempre «consulta», acierta un tercio de las veces (accuracy $\approx 0{,}33$). Pero para «consulta» tiene precisión $1/3$ y recall $1$, es decir $F_1 = 0{,}5$, y para las otras dos clases $F_1 = 0$. Su F1 macro es $(0{,}5 + 0 + 0) / 3 \approx 0{,}17$: bastante menos que el accuracy.
</div>

Para comparar todos los modelos con la misma vara usaremos la función `evaluar`, que se entrega ya escrita. Ajusta una copia del estimador en cada partición de la validación cruzada, mide el F1 macro (o la métrica que se pida) sobre la parte que dejó afuera y devuelve un `ResultadoCV`. Las particiones serán `StratifiedKFold`, que reparte los ánimos parejo entre las cinco partes. Por ahora no agrupamos por artista; en la 2.1 lo comparamos con una validación agrupada.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 ResultadoCV</strong>

  Un `ResultadoCV` guarda el resultado de una validación cruzada con estos campos: `puntajes`, un valor por partición; `promedio` y `desviacion`, la media y la desviación estándar de esos puntajes, y esta última dice cuánto cambia el resultado según la partición; `descripcion_validacion`, el esquema de validación usado, por ejemplo `StratifiedKFold(5)`, para que quede junto al número; y `promedio_entrenamiento`, el puntaje promedio sobre las canciones con que se ajustó cada partición, que usaremos en la Etapa 2 para reconocer el sobreajuste.
</div>

In [22]:
import numpy as np
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.radio_fcfm.contratos import ResultadoCV

# F1 macro con zero_division=0: un clasificador que nunca predice una clase
# no debe llenar el notebook de advertencias.
F1_MACRO = make_scorer(f1_score, average="macro", zero_division=0)


def evaluar(estimador, X, y, cv, grupos=None, scoring=F1_MACRO):
    resultado = cross_validate(
        estimador,
        X,
        y,
        cv=cv,
        groups=grupos,
        scoring=scoring,
        return_train_score=True,
    )
    puntajes = resultado["test_score"]
    return ResultadoCV(
        puntajes=puntajes.tolist(),
        promedio=float(puntajes.mean()),
        desviacion=float(puntajes.std()),
        descripcion_validacion=f"{type(cv).__name__}({cv.get_n_splits()})",
        promedio_entrenamiento=float(resultado["train_score"].mean()),
    )


log_palabras = np.log1p(
    letras_train_val["Letra"].str.count_matches(r"\S+").to_numpy()
).reshape(-1, 1)

cv_estratificada = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de ejecutar: el baseline</strong>
  1. ¿Qué accuracy esperan del baseline que responde siempre la clase más frecuente, con cuatro ánimos casi parejos?
  2. ¿Y qué F1 macro? Apliquen el razonamiento del ejemplo del call center a cuatro clases.
</div>

`Escriban sus respuestas aquí:`
1. Con cuatro ánimos casi parejos, responder siempre el más frecuente acierta cerca de 1 de cada 4 canciones: accuracy ≈ 0,25.
2. Para el ánimo que predice: precisión ≈ 0,25 y recall = 1, así que F1 = 2·0,25·1/(0,25 + 1) = 0,4. Los otros tres tienen F1 = 0. El F1 macro es ≈ 0,4/4 = 0,10, bastante menos que el accuracy, igual que en el ejemplo del call center.

Evaluemos los dos baselines: primero con accuracy y después con F1 macro.

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

resultados_estratificada = {}
for nombre, estimador, entrada in [
    ("mayoritaria", DummyClassifier(strategy="most_frequent"), textos),
    ("largo", LogisticRegression(), log_palabras),
]:
    exactitud = evaluar(
        estimador, entrada, estados, cv_estratificada, scoring="accuracy"
    )
    resultados_estratificada[nombre] = evaluar(
        estimador, entrada, estados, cv_estratificada
    )
    print(
        f"{nombre:12s} accuracy = {exactitud.promedio:.3f} | "
        f"F1 macro = {resultados_estratificada[nombre].promedio:.3f} "
        f"+/- {resultados_estratificada[nombre].desviacion:.3f}"
    )

mayoritaria  accuracy = 0.260 | F1 macro = 0.103 +/- 0.000
largo        accuracy = 0.338 | F1 macro = 0.321 +/- 0.033


El baseline del largo no lee ninguna palabra y, aun así, le gana a la clase mayoritaria. Antes de preguntarnos por qué, miremos la mediana de palabras por ánimo.

In [20]:
letras_train_val.group_by("Mood").agg(
    pl.col("Letra").str.count_matches(r"\S+").median().alias("palabras_mediana")
).sort("Mood")

Mood,palabras_mediana
str,f64
"""angry""",246.0
"""happy""",294.0
"""relaxed""",180.0
"""sad""",183.0


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Una señal que no lee</strong>
  1. Una variable que solo cuenta palabras supera a la clase mayoritaria. ¿Por qué? Apóyense en la tabla de medianas.
  2. ¿Qué advertencia le dan a Gaspar para cuando un modelo que sí lee las letras supere este baseline?
</div>

`Escriban sus respuestas aquí:`
1. Porque el largo de la letra cambia con el ánimo, la mediana es de 294 palabras en `happy` y 246 en `angry`, frente a 183 en `sad` y 180 en `relaxed`. Las canciones de alta activación tienen letras más largas que las tranquilas, así que el largo separa bastante bien la activación (aunque no la valencia). Con eso el baseline llega a un F1 macro de 0,321, frente a 0,103 de la clase mayoritaria.
2. Que superar a la clase mayoritaria no basta. Un modelo que lee también podría estar aprovechando el largo o el estilo de la letra (cuántas palabras, cuántas repeticiones) y no su significado. La comparación honesta es contra el baseline del largo: lo que el modelo gane por sobre ese ~0,30 es lo que realmente aporta leer.

Hasta acá las cuatro clases pesan lo mismo, y por eso el baseline de la clase mayoritaria es tan bajo. El catálogo real de la Fonoteca no tiene por qué estar tan parejo.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Si el catálogo estuviera desbalanceado</strong>
  1. Supongan que el catálogo de Gaspar fuera 90 % `happy`. ¿Qué accuracy y qué F1 macro tendría el baseline que responde siempre `happy`?
  2. ¿Qué cambiaría al leer el puntaje de un modelo en ese catálogo? ¿Qué métrica seguiría diciendo algo útil y cuál dejaría de hacerlo?
</div>

`Escriban sus respuestas aquí:`
1. Accuracy = 0,90. Para `happy`, precisión 0,9 y recall 1, así que F1 = 2·0,9·1/1,9 ≈ 0,947; los otros tres ánimos tienen F1 = 0. El F1 macro es ≈ 0,947/4 ≈ 0,24.
2. Un 90 % de accuracy sería el piso de no hacer nada, así que el accuracy dejaría de ser útil: un modelo que nunca reconoce `angry`, `sad` ni `relaxed` se vería excelente. El F1 macro (y la precisión y el recall por ánimo) seguiría diciendo algo, porque castiga con el mismo peso no reconocer las clases chicas. Además, cualquier puntaje habría que compararlo siempre con el baseline de la clase mayoritaria.

Los puntajes de los baselines son los primeros que vamos a querer comparar más adelante, así que los registramos en MLflow desde ya. Registren a mano el baseline de la clase mayoritaria como un run llamado `"baseline-mayoritaria_estratificada"`, con:

- los parámetros `modelo="DummyClassifier"` y `entrada="ninguna"`;
- los tags `tipo="baseline"`, `metrica="f1_macro"` y `validacion=resultados_estratificada["mayoritaria"].descripcion_validacion`;
- las métricas `promedio` y `desviacion` de `resultados_estratificada["mayoritaria"]`.

La celda de MLflow ya dejó activo el experimento `radio-fcfm`, así que el run queda ahí.

(Hint: `mlflow.start_run(run_name=...)`, `mlflow.log_params`, `mlflow.set_tags`, `mlflow.log_metrics`)

In [23]:
resultado_mayoritaria = resultados_estratificada["mayoritaria"]

# `start_run` abre el run con el nombre pedido y lo cierra al salir del `with`.
with mlflow.start_run(run_name="baseline-mayoritaria_estratificada"):
    # Parámetros: qué modelo se evaluó y con qué entrada.
    mlflow.log_params({"modelo": "DummyClassifier", "entrada": "ninguna"})
    # Tags: para filtrar después por tipo, métrica y validación.
    mlflow.set_tags(
        {
            "tipo": "baseline",
            "metrica": "f1_macro",
            "validacion": resultado_mayoritaria.descripcion_validacion,
        }
    )
    # Métricas: promedio y desviación del F1 macro entre las 5 particiones.
    mlflow.log_metrics(
        {
            "promedio": resultado_mayoritaria.promedio,
            "desviacion": resultado_mayoritaria.desviacion,
        }
    )

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 `registrar_run` y `tabla_runs` vienen implementadas</strong>
  <code>src/radio_fcfm/tracking.py</code> es material provisto: no hay que completarlo. <code>registrar_run(nombre, resultado, params, modelo=None, etiquetas=None)</code> hace lo mismo que acaban de hacer a mano, para cualquier <code>ResultadoCV</code>: registra los parámetros, la validación y las <code>etiquetas</code> como tags, el puntaje de cada fold, el promedio y la desviación (y el promedio de entrenamiento, si el resultado lo trae), y, si se le entrega, el modelo ajustado como artefacto. <code>tabla_runs()</code> devuelve los runs del experimento, uno por nombre: si ejecutan el notebook dos veces, se queda con el más reciente. Desde aquí registramos cada evaluación con estas dos funciones.
</div>

In [24]:
from src.radio_fcfm.tracking import registrar_run, tabla_runs

registrar_run(
    "baseline-largo_estratificada",
    resultados_estratificada["largo"],
    {"modelo": "LogisticRegression", "entrada": "log(1 + palabras)"},
    modelo=LogisticRegression().fit(log_palabras, estados),
    etiquetas={"tipo": "baseline", "metrica": "f1_macro"},
)

columnas_runs = [
    "tags.mlflow.runName",
    "tags.tipo",
    "tags.validacion",
    "metrics.promedio",
    "metrics.desviacion",
]
tabla_baselines = tabla_runs()
tabla_baselines[tabla_baselines["tags.tipo"] == "baseline"][columnas_runs]

,tags.mlflow.runName,tags.tipo,tags.validacion,metrics.promedio,metrics.desviacion
0,baseline-largo_estratificada,baseline,StratifiedKFold(5),0.320511,0.033160
1,baseline-mayoritaria_estratificada,baseline,StratifiedKFold(5),0.103052,0.000135


<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  La tabla debe mostrar al menos los dos runs, <code>baseline-mayoritaria_estratificada</code> y <code>baseline-largo_estratificada</code>, con F1 macro de cerca de 0,10 y 0,32, iguales a los impresos más arriba. Si ya ejecutaron el notebook antes, aparecen además los baselines agrupados de la 2.1. El accuracy de la clase mayoritaria debe quedar cerca de 0,26.
</div>

Ya que la limpieza y la partición funcionan en el notebook, vale la pena llevarlas a funciones que se puedan reutilizar y probar.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `data.py`</strong>
  Implementen en <code>src/radio_fcfm/data.py</code>:

  <ul>
    <li><code>limpiar_letra(texto)</code>: la misma limpieza de la 1.2, pero sobre un solo texto (un <code>str</code>). Quita las marcas entre corchetes, deja los paréntesis y deja un solo espacio entre palabras, sin espacios al inicio ni al final. Aplicarla dos veces da lo mismo que aplicarla una.</li>
    <li><code>separar_test(artistas, estados)</code>: la partición de la 1.3. Devuelve <code>(train_val, test)</code>, las posiciones de las canciones de cada conjunto, sin artistas en común y con los cuatro ánimos en el test. Debe ser reproducible: siempre la misma partición para los mismos datos.</li>
  </ul>
</div>

Comprobemos que las dos funciones hacen lo mismo que hicimos en el notebook. La primera parte aplica `limpiar_letra` a la letra original de cada canción y la compara con la columna `Letra` que construimos en la 1.2. La segunda llama a `separar_test` y compara las posiciones que devuelve con las que obtuvimos en la 1.3 para `train_val` y `test`.

In [25]:
from src.radio_fcfm.data import limpiar_letra, separar_test

con_funcion = letras_limpias["Lyrics"].map_elements(
    limpiar_letra, return_dtype=pl.String
)
iguales = (con_funcion == letras_limpias["Letra"]).all()
print(f"La función reproduce la limpieza: {iguales}")

train_val_f, test_f = separar_test(artistas_todos, estados_todos)
misma_particion = np.array_equal(train_val_f, train_val) and np.array_equal(
    test_f, test
)
print(f"La función reproduce la partición: {misma_particion}")

La función reproduce la limpieza: True
La función reproduce la partición: True


<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Ejecutar <code>uv run pytest -m etapa1</code>. Las dos líneas de la celda de arriba deben imprimir <code>True</code>: las funciones del módulo hacen lo mismo que lo que exploraron en el notebook.
</div>

---
## Etapa 2 — Del texto a un pipeline sin fuga (2,0 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Una consultora promete 0,56</strong>

  **Oráculo Analytics**, una consultora que Gaspar contrató antes que a ustedes, le informó un F1 macro de **0,56** para clasificar el ánimo de las canciones. Gaspar duda de una cifra tan alta y le encarga al equipo corroborarla: construir el modelo de letras y medirlo con una validación que no filtre información, averiguar de dónde sale el 0,56 y entender por qué un modelo que acierta casi todo lo que ya vio puede fallar con canciones nuevas. Por su cuenta, el equipo decide probar además con un random forest, para ver si un modelo más complejo rinde mejor.
</div>

### 2.1 — TF-IDF en un pipeline, con dos validaciones (0,5 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 TF-IDF y vocabulario</strong>

  Un modelo no lee texto: necesita números. <strong>TF-IDF</strong> representa cada texto como un vector con una coordenada por cada palabra del <strong>vocabulario</strong>. Cada coordenada multiplica dos cosas: cuánto aparece la palabra en ese texto (<em>term frequency</em>) y qué tan rara es en el conjunto de textos (<em>inverse document frequency</em>). En los mensajes de un call center, «cobro» pesa mucho en el texto de un reclamo, porque aparece en pocos mensajes y no en todos, mientras que «hola» pesa poco, porque está en casi todos.

  El vocabulario y los pesos IDF se <strong>aprenden de los textos</strong> con <code>fit</code>, y después se aplican a cualquier texto con <code>transform</code>. Eso importa: si el <code>fit</code> usa los textos de validación, el modelo sabe de antemano qué palabras existen ahí.
</div>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

mensajes = [
    "me cobraron dos veces la cuenta",
    "gracias por la atención",
    "cuánto cuesta cambiar de plan",
    "cobro duplicado en mi cuenta",
]
vectorizador_demo = TfidfVectorizer()
matriz_demo = vectorizador_demo.fit_transform(mensajes)
print(f"{matriz_demo.shape[0]} mensajes x {matriz_demo.shape[1]} palabras")
print(vectorizador_demo.get_feature_names_out())

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ TF-IDF y bolsa de palabras</strong>
  ¿En qué se diferencia TF-IDF de una **bolsa de palabras** (*bag of words*), que solo cuenta cuántas veces aparece cada palabra en la letra? ¿Qué implicancias podría tener elegir una u otra en los resultados? (Hint: `CountVectorizer`)
</div>

`Escriban su respuesta aquí:`

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Pipeline y validación agrupada</strong>

  Un <code>Pipeline</code> encadena pasos: aquí, el vectorizador y un clasificador. En la validación cruzada, scikit-learn llama <code>fit</code> sobre el pipeline completo, y solo con el entrenamiento de cada partición, así que el vocabulario de cada partición no ve sus textos de validación. Lo mismo vale para cualquier paso que aprenda de los datos.

  La validación <strong>estratificada</strong> reparte las clases parejo entre las particiones. La <strong>agrupada</strong> (<code>StratifiedGroupKFold</code>) además deja a todos los textos de un mismo grupo en la misma partición. En el call center, si un cliente escribe diez mensajes y algunos caen en entrenamiento y otros en validación, el modelo puede reconocer su estilo y sus problemas habituales, y la validación sale optimista. Agrupar por cliente imita lo que de verdad pasará: llegar un cliente que el modelo nunca vio.
</div>

Acá el grupo natural es el **artista**: un artista puede tener varias canciones, con un estilo y un vocabulario propios. Veamos cuántos artistas tienen más de una canción.

In [ ]:
conteo_artistas = letras_train_val.group_by("Artist").len()
print(f"Artistas distintos: {conteo_artistas.height}")
con_varias = conteo_artistas.filter(pl.col("len") >= 2).height
print(f"Artistas con 2 o más canciones: {con_varias}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de ejecutar: ¿qué partición puntúa más?</strong>
  1. ¿Qué validación esperan que dé un F1 macro más alto para un mismo pipeline de TF-IDF: la estratificada o la agrupada por artista? Justifiquen.
  2. Con los conteos de arriba: ¿qué importa, para la fuga de datos, que un artista tenga muchas o pocas canciones en el catálogo? ¿Esperan una diferencia grande o pequeña entre las dos validaciones?
</div>

`Escriban sus respuestas aquí:`

Construyan `pipeline_tfidf`, un `Pipeline` con dos pasos: `TfidfVectorizer(stop_words="english", min_df=2)`, de nombre `"tfidf"`, que descarta las palabras vacías del inglés y las que aparecen en menos de dos canciones, y `LogisticRegression(max_iter=1000)`, de nombre `"clasificador"`. Definan también `cv_agrupada`, un `StratifiedGroupKFold` de 5 particiones con `shuffle=True` y `random_state=0`. Evalúen el pipeline con `evaluar` con las dos validaciones, `cv_estratificada` y `cv_agrupada` (esta última con `grupos=artistas`), y guarden los resultados en `resultado_tfidf_estratificada` y `resultado_tfidf_agrupada`.

(Hint: `Pipeline`, `StratifiedGroupKFold`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan pipeline_tfidf y evalúenlo antes de continuar."
)

In [ ]:
for nombre, resultado in [
    ("estratificada", resultado_tfidf_estratificada),
    ("agrupada por artista", resultado_tfidf_agrupada),
]:
    print(
        f"TF-IDF, {nombre:21s} F1 macro = {resultado.promedio:.3f} "
        f"+/- {resultado.desviacion:.3f}"
    )

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Leer la diferencia</strong>
  1. ¿Cuál de las dos validaciones dio un F1 más alto, y por cuánto? Comparen esa diferencia con la desviación entre particiones.
  2. ¿Pueden concluir que agrupar por artista no importa en este dataset? ¿En qué otro catálogo creen que sí importaría más?
</div>

`Escriban sus respuestas aquí:`

Registremos estos resultados, junto con los baselines evaluados ahora con la misma validación agrupada, para poder compararlos más adelante. De estos runs, `tfidf_agrupada` guarda además el pipeline ajustado con todos los datos.

In [ ]:
resultados_agrupada = {}
for nombre, estimador, entrada, params in [
    (
        "mayoritaria",
        DummyClassifier(strategy="most_frequent"),
        textos,
        {"modelo": "DummyClassifier", "entrada": "ninguna"},
    ),
    (
        "largo",
        LogisticRegression(),
        log_palabras,
        {"modelo": "LogisticRegression", "entrada": "log(1 + palabras)"},
    ),
]:
    resultados_agrupada[nombre] = evaluar(
        estimador, entrada, estados, cv_agrupada, grupos=artistas
    )
    registrar_run(
        f"baseline-{nombre}_agrupada",
        resultados_agrupada[nombre],
        params,
        etiquetas={"tipo": "baseline", "metrica": "f1_macro"},
    )

parametros_tfidf = {"modelo": "LogisticRegression", "entrada": "TF-IDF"}
registrar_run(
    "tfidf_estratificada",
    resultado_tfidf_estratificada,
    parametros_tfidf,
    etiquetas={"tipo": "modelo", "metrica": "f1_macro"},
)
registrar_run(
    "tfidf_agrupada",
    resultado_tfidf_agrupada,
    parametros_tfidf,
    modelo=pipeline_tfidf.fit(textos, estados),
    etiquetas={"tipo": "modelo", "metrica": "f1_macro"},
)
print(f"Runs distintos en el experimento: {len(tabla_runs())}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  El TF-IDF estratificado debe dar un F1 macro de cerca de 0,51 y el agrupado de cerca de 0,50, los dos muy por encima del baseline del largo con la misma validación agrupada (0,30).
</div>

### 2.2 — De dónde sale el 0,56 (0,4 puntos)

La consultora explicó su método a Gaspar con una frase: «primero me quedo con las mil palabras que más se asocian al ánimo, y recién después valido el modelo». Reproduzcámoslo y comparémoslo con el procedimiento correcto.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Fuga por selección de variables</strong>

  Elegir las palabras más asociadas a la etiqueta usando <strong>todos</strong> los mensajes, y recién después separar entrenamiento y validación, deja que los mensajes de validación decidan qué palabras se eligen: la selección ya vio las etiquetas que se suponen desconocidas. Con miles de palabras y pocos mensajes, siempre hay palabras que por puro azar coinciden con las etiquetas de validación, y el modelo las aprovecha. El puntaje sale inflado, y esa ventaja desaparece cuando llegan mensajes nuevos.

  La prueba <strong>χ²</strong> mide qué tan asociada está cada palabra con la etiqueta, y <code>SelectKBest(chi2, k=...)</code> se queda con las <code>k</code> palabras de mayor puntaje. Hecha <em>dentro</em> del <code>Pipeline</code>, esa selección se recalcula en cada partición solo con su entrenamiento.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de ejecutar: fuera o dentro</strong>
  1. ¿Qué versión esperan que dé un F1 más alto: seleccionar las palabras con todo el corpus antes de validar, o hacerlo dentro del pipeline?
  2. Si en vez de 1 000 palabras se eligen 200, ¿esperan que la ventaja de la versión de fuera crezca, se mantenga o se achique?
</div>

`Escriban sus respuestas aquí:`

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ La versión «fuera» está mal a propósito</strong>
  Seleccionar las palabras con todo el corpus antes de validar es justo la fuga que acabamos de definir. Se hace así solo para medir cuánto infla el puntaje: en cualquier otro punto del laboratorio, la selección va dentro del <code>Pipeline</code>.
</div>

Construyan `resultados_seleccion`, un diccionario con las claves `(k, "fuera")` y `(k, "dentro")` para `k` en `[200, 1000]`, cada una con un `ResultadoCV` evaluado con `cv_agrupada` y `grupos=artistas`:

- **fuera**: vectoricen todas las letras de `textos` (train y validation; el test no se toca) con `TfidfVectorizer(stop_words="english", min_df=2)`, elijan `k` palabras con `SelectKBest(chi2, k=k)` sobre esa matriz completa, con `estados` como etiqueta, y evalúen una `LogisticRegression(max_iter=1000)` sobre la matriz ya seleccionada.
- **dentro**: un `Pipeline` con el vectorizador, la selección y la regresión logística, evaluado sobre `textos`.

(Hint: `SelectKBest`, `chi2`, `fit_transform`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan resultados_seleccion antes de continuar."
)

In [ ]:
for (k, donde), resultado in resultados_seleccion.items():
    print(
        f"{k:5d} palabras, selección {donde:6s} del pipeline: "
        f"F1 macro = {resultado.promedio:.3f}"
    )

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ El 0,56 de la consultora</strong>
  1. ¿Qué fila de la tabla corresponde a lo que informó la consultora? ¿Por qué ese número no le sirve a Gaspar para decidir?
  2. ¿Qué número honesto le darían a Gaspar con estas mil palabras? Compárenlo con el TF-IDF sin selección de la 2.1.
</div>

`Escriban sus respuestas aquí:`

Registremos estas cuatro evaluaciones. Las de fuera quedan marcadas con el tag `fuga`, para que no se confundan con un modelo que se pueda usar.

In [ ]:
for (k, donde), resultado in resultados_seleccion.items():
    registrar_run(
        f"tfidf_seleccion-{k}_{donde}",
        resultado,
        {"modelo": "LogisticRegression", "entrada": f"TF-IDF + chi2({k})"},
        etiquetas={
            "tipo": "fuga" if donde == "fuera" else "exploracion",
            "metrica": "f1_macro",
        },
    )

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Con 200 palabras, la versión de fuera debe dar cerca de 0,51 y la de dentro cerca de 0,46. Con 1 000, cerca de 0,56 y 0,48. En los dos casos, fuera supera a dentro.
</div>

### 2.3 — Regularización y sobreajuste (0,5 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Sobreajuste, regularización y el parámetro C</strong>

  Un clasificador lineal asigna un peso a cada palabra del vocabulario. Con miles de palabras y relativamente pocos mensajes, puede ajustar pesos enormes que explican hasta el ruido de los mensajes de entrenamiento y fallan con mensajes nuevos: eso es el **sobreajuste**. Se reconoce porque el puntaje de **entrenamiento** (sobre los mensajes con que se ajustó el modelo) sigue subiendo mientras el de **validación** (sobre los que quedaron afuera) se estanca o baja.

  La **regularización** castiga los pesos grandes. En `LogisticRegression` se controla con `C`, que funciona *al revés*: un `C` **bajo** regulariza **más**. Un call center con muchas palabras y pocos mensajes necesita regularizar más que uno con muchísimos más mensajes que palabras, pero regularizar de más tampoco sirve: con pesos demasiado castigados el modelo no alcanza a aprender (**subajuste**).

  El `C` es un **hiperparámetro**: no lo aprende el modelo, lo elige quien lo usa. Si se elige probando varios valores y quedándose con el que da el mejor puntaje de validación, ese puntaje ya no es una medición independiente: se usó para decidir. Queda un poco optimista.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de ejecutar: de C = 0,01 a C = 100</strong>
  1. Si `C` sube de 0,01 a 100, ¿qué esperan que le pase al F1 de **entrenamiento**?
  2. ¿Y al F1 de **validación**? ¿Esperan que siga el mismo camino?
</div>

`Escriban sus respuestas aquí:`

Veamos qué pasa con cinco valores de `C`, con el pipeline de la 2.1. La lista `valores_C` se puede editar: cambien los valores y vuelvan a ejecutar la celda. Cada evaluación queda registrada como una exploración, con su puntaje de entrenamiento.

In [ ]:
from sklearn.pipeline import Pipeline

valores_C = [0.01, 0.1, 1.0, 10.0, 100.0]

for C in valores_C:
    # El mismo pipeline de la 2.1, con otro valor de C.
    pipeline_C = Pipeline(
        [
            ("tfidf", TfidfVectorizer(stop_words="english", min_df=2)),
            ("clasificador", LogisticRegression(C=C, max_iter=1000)),
        ]
    )
    resultado_C = evaluar(
        pipeline_C, textos, estados, cv_agrupada, grupos=artistas
    )
    registrar_run(
        f"tfidf-C{C}_exploracion",
        resultado_C,
        {"modelo": "LogisticRegression", "entrada": "TF-IDF", "C": C},
        etiquetas={"tipo": "exploracion", "metrica": "f1_macro"},
    )
    brecha = resultado_C.promedio_entrenamiento - resultado_C.promedio
    print(
        f"C = {C:<6} entrenamiento = {resultado_C.promedio_entrenamiento:.3f}"
        f" | validación = {resultado_C.promedio:.3f}"
        f" | brecha = {brecha:.3f}"
    )

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Elegir el C</strong>
  1. ¿Qué `C` elegiría alguien que mirara solo el F1 de entrenamiento? ¿Cómo le iría con canciones nuevas, comparado con el mejor `C` según la validación?
  2. ¿Qué `C` elegirían ustedes? ¿Por qué el puntaje de ese `C` es optimista, y qué harían para estimar mejor el desempeño con él?
</div>

`Escriban sus respuestas aquí:`

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  El F1 de entrenamiento debe subir con <code>C</code>, de cerca de 0,18 con <code>C=0.01</code> a cerca de 1,0 con <code>C=100.0</code>. El de validación debe partir en cerca de 0,16 con <code>C=0.01</code>, pasar por un máximo de cerca de 0,50 con <code>C=1.0</code> y bajar a cerca de 0,47 con <code>C=100.0</code>.
</div>

### 2.4 — Un árbol frente a un bosque (0,6 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Árbol de decisión y bosque aleatorio</strong>

  Un **árbol de decisión** clasifica con una cadena de preguntas de sí o no sobre las palabras: «¿aparece *cobro*?», y si sí, «¿aparece *duplicado*?». Si no se limita su profundidad, sigue preguntando hasta que cada mensaje de entrenamiento queda solo en su hoja: acierta todo lo que ya vio, y por eso sobreajusta con facilidad.

  Un **bosque aleatorio** entrena muchos árboles y los hace **votar**. Cada árbol se ajusta con una muestra distinta de los mensajes (sorteados con reemplazo) y, en cada pregunta, solo considera un subconjunto de palabras. Así los árboles se equivocan en mensajes distintos, y al votar esos errores se cancelan en parte. Esta idea, combinar muchos modelos distintos para reducir el ruido de cada uno, es un **ensamblaje**. Otra familia, el *boosting*, no los hace votar sino que entrena los árboles en secuencia, cada uno corrigiendo los errores del anterior; aquí no la usaremos.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de ejecutar: el árbol y el bosque</strong>
  1. Un árbol de decisión sin límite de profundidad, ¿qué F1 esperan en entrenamiento y cuál en validación?
  2. Un bosque de 100 árboles igual de profundos, ¿esperan que rinda más o menos que un árbol solo en validación? ¿Y que la regresión logística de la 2.1?
</div>

`Escriban sus respuestas aquí:`

Construyan `pipeline_arbol` y `pipeline_bosque`: copias sin ajustar de `pipeline_tfidf` cuyo paso `"clasificador"` es un `DecisionTreeClassifier(random_state=0)` y un `RandomForestClassifier(n_estimators=100, random_state=0)`. Así el TF-IDF y la validación son idénticos a los de la regresión logística, y lo único que cambia es el clasificador. Evalúenlos con `evaluar`, con `cv_agrupada` y `grupos=artistas`, y guarden los resultados en `resultado_arbol` y `resultado_bosque`.

(Hint: `clone`, `set_params`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan pipeline_arbol y pipeline_bosque antes de continuar."
)

In [ ]:
for nombre, resultado in [
    ("Regresión logística", resultado_tfidf_agrupada),
    ("Árbol sin límite", resultado_arbol),
    ("Bosque de 100", resultado_bosque),
]:
    print(
        f"{nombre:20s} entrenamiento = {resultado.promedio_entrenamiento:.3f}"
        f" | validación = {resultado.promedio:.3f} ± {resultado.desviacion:.3f}"
    )

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ El árbol y el bosque</strong>
  1. ¿Por qué el árbol saca casi 1 en entrenamiento y tan poco en validación?
  2. El bosque tiene árboles igual de profundos y también saca casi 1 en entrenamiento. ¿Por qué aun así rinde más en validación que un árbol solo?
  3. ¿Superó el bosque a la regresión logística? ¿Qué le dirían a Gaspar sobre usar un modelo más complejo?
</div>

`Escriban sus respuestas aquí:`

Registremos las dos evaluaciones. El bosque queda como modelo candidato, con su pipeline ajustado con todos los datos; el árbol sin límite es solo una exploración, y no se guarda.

In [ ]:
registrar_run(
    "arbol_agrupada_exploracion",
    resultado_arbol,
    {"modelo": "DecisionTreeClassifier", "entrada": "TF-IDF"},
    etiquetas={"tipo": "exploracion", "metrica": "f1_macro"},
)
registrar_run(
    "bosque_agrupada",
    resultado_bosque,
    {"modelo": "RandomForestClassifier", "entrada": "TF-IDF"},
    modelo=pipeline_bosque.fit(textos, estados),
    etiquetas={"tipo": "modelo", "metrica": "f1_macro"},
)
print(f"Runs distintos en el experimento: {len(tabla_runs())}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  El árbol debe dar un F1 de entrenamiento de cerca de 1,0 y uno de validación de cerca de 0,36. El bosque, cerca de 1,0 y 0,48. La regresión logística, cerca de 0,93 y 0,50.
</div>

Con el pipeline probado en el notebook, llevémoslo a una función reutilizable.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `pipeline.py`</strong>
  Implementen en <code>src/radio_fcfm/pipeline.py</code>:

  <ul>
    <li><code>construir_pipeline(k_palabras=None)</code>: el pipeline de la 2.1, sin ajustar, con los pasos <code>"tfidf"</code> y <code>"clasificador"</code> (<code>"tfidf"</code> con <code>stop_words="english"</code> y <code>min_df=2</code>; <code>"clasificador"</code> con <code>max_iter=1000</code>). Si <code>k_palabras</code> no es <code>None</code>, inserta entre los dos un paso <code>"seleccion"</code>: el <code>SelectKBest(chi2, k=k_palabras)</code> de la 2.2.</li>
  </ul>
</div>

In [ ]:
from src.radio_fcfm.pipeline import construir_pipeline

f1_tfidf_modulo = evaluar(
    construir_pipeline(), textos, estados, cv_agrupada, grupos=artistas
)
f1_seleccion_modulo = evaluar(
    construir_pipeline(k_palabras=200),
    textos,
    estados,
    cv_agrupada,
    grupos=artistas,
)

dentro_200 = resultados_seleccion[(200, "dentro")]
comparaciones = [
    ("TF-IDF", f1_tfidf_modulo, resultado_tfidf_agrupada),
    ("TF-IDF + 200", f1_seleccion_modulo, dentro_200),
]
for nombre, desde_modulo, desde_exploracion in comparaciones:
    print(
        f"{nombre:13s} módulo {desde_modulo.promedio:.3f} | "
        f"exploración {desde_exploracion.promedio:.3f}"
    )

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Ejecutar <code>uv run pytest -m etapa2</code>. Las dos líneas de F1 deben mostrar el mismo valor en el módulo y en la exploración.
</div>

---
## Etapa 3 — Métricas y errores (1,5 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 "No quiero una canción furiosa a las ocho de la mañana"</strong>

  Gaspar ya sabe de dónde salía el 0,56 de la consultora, y tiene un número honesto: unos 0,50 con el modelo de letras, que ni el random forest logró superar. Pero no se conforma. «Que acierte la mitad de las veces no me dice nada. Lo que me quita el sueño es poner una canción furiosa a las ocho de la mañana. ¿Dónde se equivoca el modelo? ¿Y cuántas veces se equivoca en lo que me importa, que es la franja?».
</div>

### 3.1 — Precisión, recall y F1 por ánimo (0,3 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Precisión y recall en un ejemplo, y predicciones fuera de muestra</strong>

  Un call center marca 40 mensajes como «reclamo» y 30 lo eran: su precisión es $30/40 = 0{,}75$. Había 50 reclamos en total y encontró esos 30: su recall es $30/50 = 0{,}60$. Con precisión baja se molesta a clientes que no reclamaron; con recall bajo, se pierden reclamos reales.

  Para calcularlas hay que mirar predicciones **fuera de muestra**: la predicción de cada canción debe venir de un modelo que no la vio al ajustarse. `cross_val_predict` hace eso: reparte las canciones en particiones, ajusta con las demás y predice la que dejó afuera.
</div>

Construyan `pred_tfidf`: la predicción de cada canción de train y validation cuando le tocó estar en la partición de validación, con `construir_pipeline()`, de `src/radio_fcfm/`, y las letras de `textos`. Usen `cv_agrupada` y `groups=artistas` (en `cross_val_predict` el parámetro se llama `groups`, no `grupos` como en `evaluar`). Debe tener 1483 predicciones, en el mismo orden que `estados`.

(Hint: `cross_val_predict`)

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan pred_tfidf antes de continuar.")

Veamos el reporte por ánimo.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(estados, pred_tfidf, digits=3))

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Leer el reporte</strong>
  1. ¿Qué ánimo se reconoce mejor (recall) y cuál peor? ¿Qué le dirían a Gaspar sobre las canciones de cada franja?
  2. La precisión de `happy` es la que más le importa a Gaspar para la mañana. ¿Qué significa ese número en palabras?
</div>

`Escriban sus respuestas aquí:`

### 3.2 — La matriz de confusión y los dos ejes (0,5 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Matriz de confusión</strong>

  La **matriz de confusión** cuenta, para cada clase real (filas), cuántos ejemplos se predijeron como cada clase (columnas). La diagonal son los aciertos; todo lo demás, errores, y cada celda dice *qué se confundió con qué*. En un call center, si muchos «reclamos» se predicen como «consultas» y casi ninguno como «felicitaciones», el modelo se equivoca sobre todo entre categorías cercanas, y eso se ve en la matriz aunque el accuracy no lo muestre.

  Esa información crece cuando las categorías ocupan un cuadrante de dos ejes. Un call center que clasifica los mensajes por tipo (reclamo o consulta) y por urgencia (urgente o corriente) tiene cuatro categorías: *reclamo urgente*, *reclamo corriente*, *consulta urgente* y *consulta corriente*. Confundir «reclamo urgente» con «reclamo corriente» cambia **un solo eje**, la urgencia; confundirlo con «consulta corriente» cambia los dos. Los cuatro ánimos de este laboratorio tienen la misma estructura, con los ejes de valencia y activación que ya vimos.
</div>

In [ ]:
import plotly.express as px
from sklearn.metrics import confusion_matrix

from src.radio_fcfm.contratos import ACTIVACION, ESTADOS, VALENCIA

matriz = confusion_matrix(estados, pred_tfidf, labels=ESTADOS)
px.imshow(
    matriz,
    x=ESTADOS,
    y=ESTADOS,
    text_auto=True,
    labels={"x": "Predicho", "y": "Real", "color": "Canciones"},
    title="Matriz de confusión, TF-IDF (agrupada por artista)",
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de contar los errores por eje</strong>
  1. Mirando la matriz, ¿qué par de ánimos se confunde más? ¿Comparten algún eje?
  2. Si los errores se repartieran al azar entre los tres ánimos equivocados posibles, un tercio cambiaría los dos ejes. ¿Esperan que en la matriz real sean más, menos o parecido a un tercio? ¿Por qué?
</div>

`Escriban sus respuestas aquí:`

Construyan `errores_por_eje`, un diccionario con tres claves, `"solo valencia"`, `"solo activación"` y `"ambos"`, que cuente cuántas de las predicciones equivocadas de `pred_tfidf` cambian solo la valencia, solo la activación o las dos. Usen `VALENCIA` y `ACTIVACION`, dos diccionarios que dan la valencia y la activación de cada ánimo. No usen `for` sobre las canciones: armen una tabla de Polars con `real` y `predicho` y resuélvanlo con expresiones, que recorren las 1483 filas en código compilado y son lo idiomático para mapear una columna.

(Hint: `replace_strict`, `filter`)

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan errores_por_eje antes de continuar.")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Los tres valores de <code>errores_por_eje</code> deben sumar los errores totales, 736 (las 1483 canciones menos los 747 aciertos de la diagonal), repartidos en cerca de 316 solo de valencia, 272 solo de activación y 148 de ambos.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Dónde se equivoca el modelo</strong>
  1. ¿Qué fracción de los errores cambia los dos ejes? Compárenla con el tercio del azar.
  2. Los errores de un solo eje, ¿son más graves o menos graves para las franjas de Gaspar (mañana: `happy`; hora del taco: `angry`; noche: `sad` y `relaxed`)? Piensen en `sad` y `relaxed`, y en `happy` y `angry`.
</div>

`Escriban sus respuestas aquí:`

### 3.3 — Franjas y la métrica para Gaspar (0,4 puntos)

Gaspar no programa cuatro ánimos, programa tres franjas: la mañana (`happy`), la hora del taco (`angry`) y la noche (`sad` y `relaxed` juntos). Un error entre dos ánimos de la misma franja no lo afecta. `FRANJAS`, en `src/radio_fcfm/contratos.py`, es un diccionario que dice a qué franja va cada ánimo.

Construyan `acierto_franja`, la fracción de canciones de train y validation cuya franja predicha en `pred_tfidf` coincide con la real. Hagan el mapeo con expresiones de Polars, por la misma razón que en la 3.2. Debe dar cerca de 0,641.

(Hint: `replace_strict`)

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan acierto_franja antes de continuar.")

In [ ]:
from src.radio_fcfm.contratos import FRANJAS

acierto_cuatro = float((estados == pred_tfidf).mean())
piso_noche = float((pl.Series(estados).replace_strict(FRANJAS) == "noche").mean())
print(
    f"Cuatro ánimos: {acierto_cuatro:.3f} | "
    f"tres franjas: {acierto_franja:.3f}"
)
print(f"Responder siempre «noche»: {piso_noche:.3f}")

Veamos también qué le llegaría de verdad a la franja de la mañana: de todas las canciones que el modelo manda ahí, ¿qué ánimo tenían en realidad?

In [ ]:
(
    pl.DataFrame({"real": estados, "predicho": pred_tfidf})
    .filter(pl.col("predicho") == "happy")
    .group_by("real")
    .len()
    .sort("len", descending=True)
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Qué número le reportan a Gaspar</strong>
  1. ¿Por qué el acierto por franja es mayor que el accuracy de los cuatro ánimos? ¿Podría ser menor?
  2. De las canciones que irían a la mañana, ¿cuántas no eran alegres, y cuántas eran de `angry`? ¿Qué fracción de la franja son estas últimas?
  3. ¿Qué métrica o métricas le reportarían a Gaspar: el accuracy de los cuatro ánimos, el acierto por franja o la precisión de `happy`? Justifiquen con lo que a él le importa.
</div>

`Escriban sus respuestas aquí:`

### 3.4 — Qué palabras pesan, y qué no es emoción (0,3 puntos)

TF-IDF con regresión logística no es una caja negra: cada palabra tiene un peso por ánimo, y las de mayor peso dicen en qué se apoya el modelo. Construyan `palabras_por_estado`, un diccionario de cada ánimo a sus 8 palabras de mayor peso, ajustando `construir_pipeline()` con todos los datos. Recuerden que `"tfidf"` y `"clasificador"` son los nombres de sus pasos. Deben quedar cuatro ánimos con ocho palabras cada uno.

(Hint: `get_feature_names_out`, `coef_`, `np.argsort`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan palabras_por_estado antes de continuar."
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Palabras y atajos</strong>
  1. Identifiquen al menos una palabra de las listas que no exprese emoción. ¿Qué riesgo implica para Gaspar que el modelo se apoye en ella?
  2. La etiqueta viene de los tags de oyentes de Last.fm, que describen la canción completa: letra, sonido y género. ¿Qué significa eso para el techo de un modelo que solo lee la letra?
</div>

`Escriban sus respuestas aquí:`

Las dos medidas por franja ya salen en el notebook; las pasamos a `evaluation.py` para reutilizarlas y probarlas.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `evaluation.py`</strong>
  Implementen en <code>src/radio_fcfm/evaluation.py</code>:

  <ul>
    <li><code>a_franja(estado)</code>: la franja de programación de un ánimo, según <code>FRANJAS</code>. Si el ánimo no es uno de los cuatro, levanta un <code>ValueError</code> que nombra el ánimo desconocido.</li>
    <li><code>acierto_por_franja(y_real, y_pred)</code>: la fracción de canciones cuya franja predicha es la correcta. Compara franjas, no ánimos, y levanta un <code>ValueError</code> si <code>y_real</code> e <code>y_pred</code> tienen largo distinto.</li>
  </ul>
</div>

In [ ]:
from src.radio_fcfm.evaluation import a_franja, acierto_por_franja

print(
    f"TF-IDF módulo {acierto_por_franja(estados, pred_tfidf):.3f} | "
    f"exploración {acierto_franja:.3f}"
)
print(f"Franja de 'angry': {a_franja('angry')}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Ejecutar <code>uv run pytest -m etapa3</code>. La primera línea debe mostrar el mismo acierto en el módulo y en la exploración (0,641), y la segunda debe imprimir <code>hora del taco</code>.
</div>

---
## Etapa 4 — Decisión, test, Jev y canciones propias (1,5 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 ¿Qué haría Gaspar ahora?</strong>

  Gaspar ya escuchó a la consultora y al equipo, y entiende dónde se equivoca el modelo: acierta la franja en dos de cada tres canciones, y de cada diez que manda a la mañana más de cuatro no son alegres. Ahora tiene que decidir qué hacer con el catálogo de la Fonoteca y no sabe por dónde partir: dejar funcionando alguno de los modelos, probar algo distinto o pedir más información antes de comprometerse. Le pide al equipo que revise lo que quedó registrado y le diga qué le recomendarían hacer, con los números a la vista.

  La Fonoteca, por su parte, no firma sin ver cómo le va al modelo con los artistas que se guardaron. Y Gaspar tiene una última petición, más personal: probarlo con canciones de su propia playlist, «para ver si me creo lo que dice».
</div>

### 4.1 — La tabla de runs (0,3 puntos)

Todo lo que evaluamos quedó en MLflow. Consultemos la tabla completa, ordenada de mayor a menor F1 macro.

In [ ]:
tabla = tabla_runs()
tabla[columnas_runs].sort_values("metrics.promedio", ascending=False)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ El mejor puntaje de la tabla</strong>
  1. ¿Deberían recomendarle a Gaspar el run con mayor puntaje de toda la tabla? Revisen su `tipo` y su `validacion`.
  2. ¿Qué tienen en común los runs que le siguen y quedan más arriba en la tabla?
</div>

`Escriban sus respuestas aquí:`

Para decidir hay que quedarse con los runs que sirven para eso. Antes de filtrar, justifiquen cuáles.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Qué runs pueden decidir</strong>
  1. Con lo que acaban de ver en la tabla, ¿qué condiciones debe cumplir un run para que se pueda decidir con él? Piensen en las columnas `tags.tipo` y `tags.validacion`.
  2. ¿Cuántos runs esperan que queden al aplicarlas?
</div>

`Escriban sus respuestas aquí:`

Construyan `tabla_decision`: de `tabla`, los runs cuyo `tags.tipo` es `modelo` o `baseline` y cuya `tags.validacion` empieza con `StratifiedGroupKFold`, ordenados de mayor a menor `metrics.promedio`. Deben quedar 4 filas.

(Hint: `isin`, `str.startswith`, `sort_values`)

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan tabla_decision antes de continuar.")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  <code>tabla</code> debe tener una fila por nombre de run, aunque hayan ejecutado el notebook más de una vez. Si ya llegaron a la 4.3 en una ejecución anterior, aparece además <code>jev_test</code>, de tipo <code>externo</code>. <code>tabla_decision</code> debe tener cuatro filas: <code>tfidf_agrupada</code>, <code>bosque_agrupada</code> y los dos baselines agrupados, en ese orden.
</div>

### 4.2 — Recargar un run y mirar el test (0,3 puntos)

El modelo que encabeza la tabla de decisión quedó guardado con su pipeline ajustado con todos los datos. Recarguémoslo y comprobemos que predice lo mismo que el pipeline de la Etapa 2 ajustado de nuevo.

In [ ]:
import mlflow.sklearn

fila_elegida = tabla_decision[tabla_decision["tags.tipo"] == "modelo"].iloc[0]
print(f"Run elegido: {fila_elegida['tags.mlflow.runName']}")
modelo_recargado = mlflow.sklearn.load_model(
    f"runs:/{fila_elegida['run_id']}/modelo"
)
modelo_de_nuevo = construir_pipeline().fit(textos, estados)
iguales = (
    modelo_recargado.predict(textos) == modelo_de_nuevo.predict(textos)
).all()
print(f"Predicciones iguales a las del pipeline ajustado de nuevo: {iguales}")

Ahora sí, el test: la primera vez que se usan los artistas apartados en la 1.3, con el modelo ya elegido.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de abrir el test</strong>
  1. ¿Qué F1 macro esperan en el test, comparado con el de validación del run elegido: más alto, más bajo o parecido? ¿Por qué?
  2. El test tiene 372 canciones. ¿Cuánto esperan que se mueva el accuracy del test solo por azar? (Hint: el error estándar de una proporción $p$ medida en $n$ casos es $\sqrt{p(1-p)/n}$.)
</div>

`Escriban sus respuestas aquí:`

In [ ]:
from sklearn.metrics import accuracy_score

pred_test = modelo_recargado.predict(textos_test)
f1_test = f1_score(estados_test, pred_test, average="macro")
exactitud_test = accuracy_score(estados_test, pred_test)
franja_test = acierto_por_franja(estados_test, pred_test)
validacion_elegida = fila_elegida["metrics.promedio"]

print(f"Test: {len(textos_test)} canciones de artistas nuevos")
print(f"  F1 macro           = {f1_test:.3f}")
print(f"  accuracy           = {exactitud_test:.3f}")
print(f"  acierto por franja = {franja_test:.3f}")
print(f"(validación del run elegido: F1 macro = {validacion_elegida:.3f})")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Qué dice el test</strong>
  1. Comparen el F1 del test con el de validación. ¿Se confirma lo que midió la validación agrupada?
  2. ¿Qué le prometen a Gaspar: un número exacto o un rango? ¿Cuál? (Construyan el rango con la cifra del test y su margen de $\pm 0{,}05$.)
  3. Si el resultado no les gustara, ¿podrían probar otro modelo y volver a mirar el test? ¿Por qué?
</div>

`Escriban sus respuestas aquí:`

### 4.3 — Un amigo con Jev (0,3 puntos)

Un amigo de Gaspar le asegura que su problema se resuelve con una tecnología nueva, llamada **Jev**, sin entrenar nada. Gaspar les pide que averigüen cómo funciona y si de verdad lo haría mejor. Empiecen por el [quickstart de Jev](https://docs.typesafe.ai/introduction/quickstart).

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Un modelo de lenguaje por API, sin entrenar</strong>

  Un call center que quiere derivar mensajes a *reclamo*, *consulta* o *felicitación* podría, en vez de entrenar un clasificador con miles de mensajes etiquetados, enviar cada mensaje a un servicio en la nube y preguntarle «¿a cuál de estas categorías corresponde?», describiendo cada opción con palabras. No hay entrenamiento ni etiquetas propias: el modelo responde con lo que ya aprendió antes, lo que se llama *zero-shot* («sin ejemplos»).

  A cambio, cada mensaje es una **llamada a un servicio externo**: cuesta por llamada, tarda, saca el texto de la organización y, si el modelo se llama `latest`, puede cambiar sin aviso. Jev, de TypeSafe, funciona así: se le entrega un texto y una pregunta de opción múltiple (`Choice`), y devuelve la opción elegida y la confianza de esa opción.

  El quickstart de arriba usa la dirección de TypeSafe. En este laboratorio la llamada va a **OpenRouter**, que recibe las llamadas con la clave del curso, y el modelo es `jev-latest` (`MODELO`, en `src/radio_fcfm/jev.py`). La celda de la clave deja configurada esa dirección.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Antes de probarlo</strong>
  1. Con lo que leyeron en la documentación, ¿en qué se diferencia Jev de lo que hicimos hasta ahora con TF-IDF y la regresión logística?
  2. ¿Esperan que clasifique mejor o peor que la regresión logística en el test? ¿Por qué?
</div>

`Escriban sus respuestas aquí:`

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 La clave y las letras</strong>
  La clave de OpenRouter la entrega el equipo docente: se ingresa con <code>getpass</code> cuando el notebook la pide, y no se escribe en el notebook, en un archivo ni en la Pull Request. Las letras del test se envían a OpenRouter y al proveedor que atienda el modelo, y son solo para este curso (ver <code>data/raw/README.md</code>): no las usen con ningún otro servicio. La primera vez son 372 llamadas, y las respuestas quedan guardadas en <code>data/processed/jev_test.csv</code> para no repetirlas.
</div>

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ El prompt de Jev queda fijo</strong>
  Las instrucciones y los criterios que se le entregan a Jev están en <code>src/radio_fcfm/jev.py</code> y no se tocan. Si los ajustan mirando el resultado en el test, el test deja de ser independiente, por la misma razón que en la 4.2.
</div>

In [ ]:
import getpass
import os

from src.radio_fcfm.jev import MODELO, URL_OPENROUTER

os.environ["TYPESAFE_BASE_URL"] = URL_OPENROUTER
if not os.environ.get("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("Clave de OpenRouter: ")

In [ ]:
from typesafe_sdk import TypeSafeClient

cliente = TypeSafeClient(model=MODELO)

Construyan `preguntar_a_jev(letra)`: recibe una letra y devuelve `(animo, confianza)`, el ánimo que elige Jev y la confianza de esa opción. Usen `cliente`, una pregunta `Choice` con las `INSTRUCCIONES` y los `CRITERIOS` de `src/radio_fcfm/jev.py` (la clave de la pregunta es `"animo"`), y recorten la letra a `MAX_CARACTERES` caracteres.

(Hint: `Choice`, `system_one`, `choices`)

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan preguntar_a_jev antes de continuar.")

Probémosla con una canción de train y validation, para no tocar el test todavía.

In [ ]:
animo, confianza = preguntar_a_jev(textos[0])
print(
    f"Jev dice {animo} (confianza {confianza:.2f}); "
    f"el ánimo real es {estados[0]}"
)

Ahora las canciones del test. `clasificar_con_jev` (material provisto en `src/radio_fcfm/jev.py`) llama a `preguntar_a_jev` para cada letra, con varios hilos, y guarda las respuestas para no repetir las llamadas. Si ya ejecutaron esta celda, lee el archivo y no llama a la API; borren `data/processed/jev_test.csv` para volver a llamarla.

In [ ]:
from src.radio_fcfm.jev import clasificar_con_jev

ruta_jev = RAW_DIR.parent / "processed" / "jev_test.csv"
animos_jev, confianza_jev = clasificar_con_jev(
    textos_test, preguntar_a_jev, ruta_cache=ruta_jev
)

Con las respuestas válidas de Jev (`animos_jev` que no es `None`), construyan `real_jev` y `pred_jev` (los ánimos reales y los que predijo Jev), y con ellos `f1_jev` y `franja_jev`: el F1 macro y el acierto por franja sobre el test, con las mismas funciones de la Etapa 3. Después registren un run llamado `"jev_test"` con `registrar_run`: un `ResultadoCV` con `puntajes=[f1_jev]`, `promedio=f1_jev`, `desviacion=0.0` (es una sola medición, sin particiones entre las que medir una desviación) y una `descripcion_validacion` que diga cuántas canciones del test se usaron y que no hubo entrenamiento; los parámetros `modelo`, `entrada` y `servicio`; y las etiquetas `tipo="externo"` y `metrica="f1_macro"`. No hay `modelo` que guardar: Jev vive en la API.

(Hint: `np.array`, `f1_score`, `acierto_por_franja`, `ResultadoCV`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan f1_jev y registren el run jev_test antes de continuar."
)

Comparemos a Jev con la regresión logística, sobre las mismas canciones del test y con las mismas métricas, y miremos dónde se equivoca Jev.

In [ ]:
print(f"{'':22s} F1 macro   acierto por franja")
print(f"{'Regresión logística':22s} {f1_test:.3f}      {franja_test:.3f}")
print(f"{'Jev (sin entrenar)':22s} {f1_jev:.3f}      {franja_jev:.3f}")

print(f"Canciones del test que respondió Jev: {len(real_jev)} de {len(animos_jev)}")
print("Matriz de Jev: filas = ánimo real, columnas = ánimo predicho por Jev")
matriz_jev = confusion_matrix(real_jev, pred_jev, labels=ESTADOS)
(
    pl.DataFrame(matriz_jev, schema=ESTADOS, orient="row")
    .with_columns(pl.Series("real", ESTADOS))
    .select("real", *ESTADOS)
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  La celda de las 372 canciones debe terminar con <code>Respuestas válidas: 372 de 372</code> (o muy cerca), y <code>tabla_runs()</code> debe mostrar el run <code>jev_test</code>, con tipo <code>externo</code>. La matriz de Jev tiene una fila por ánimo real y una columna por ánimo predicho, en el orden <code>happy</code>, <code>relaxed</code>, <code>sad</code>, <code>angry</code>; sus celdas deben sumar las canciones que respondió Jev.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Qué dice Jev frente a TF-IDF</strong>
  1. Comparen el F1 macro y el acierto por franja de Jev con los de la regresión logística, en el test. ¿Cuál rindió más? ¿La diferencia supera el margen de $\pm 0{,}05$ del test?
  2. Mirando la matriz de Jev, ¿clasifica mejor la valencia o la activación? ¿Qué ánimos confunde más y qué eje cambian esos errores? ¿Pasa lo mismo con la regresión logística de la 3.2?
  3. Además del puntaje, ¿qué más le tendrían que decir a Gaspar para decidir entre los dos? Piensen en el costo, la latencia, lo que pasa con las letras y qué ocurre si `jev-latest` cambia.
</div>

`Escriban sus respuestas aquí:`

### 4.4 — Canciones propias (0,3 puntos)

Hasta acá probamos el modelo con canciones del catálogo de referencia. Ahora lo usaremos, con el mejor modelo que obtuvimos hasta el momento, el que recargamos en la 4.2, y con canciones que ustedes elijan: canciones que les gusten, fuera de los datos con que contamos. No es una evaluación: unas pocas canciones no representan al catálogo ni alcanzan para medir un acierto. Es un ejercicio para hacerse una idea de cómo funciona el modelo y, también, por curiosidad: ¿qué ánimo le ve a su canción favorita?

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Solo canciones en inglés, y sin dejar la letra en la Pull Request</strong>
  El modelo se entrenó únicamente con letras en inglés: con otro idioma, la predicción no significa nada. Las letras tienen derechos de autor: antes de hacer <code>commit</code>, vuelvan a dejar en la celda la letra de ejemplo, para que no suban ninguna canción al repositorio.
</div>

Peguen la letra de una canción entre las comillas de `letra` y escriban su `titulo`.

In [ ]:
titulo = "Summer Sun (example lyrics)"
# La letra debe estar en inglés: el modelo se entrenó solo con letras en inglés.
letra = """
Sunshine on the street, we are dancing in the light, oh yeah,
everybody sing along, we are happy tonight, la la la,
the summer never ends
"""

Construyan `animo`, `franja` y `confianza` para esa letra: límpienla con `limpiar_letra`, pídanle al `modelo_recargado` la probabilidad de cada ánimo y quédense con el de mayor probabilidad. La `franja` es la que le corresponde a ese ánimo, y la `confianza`, su probabilidad, un número entre 0 y 1.

(Hint: `predict_proba`, `classes_`, `argmax`, `a_franja`)

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan animo, franja y confianza antes de continuar."
)

In [ ]:
print(f"{titulo}: {animo} (franja: {franja}), confianza {confianza:.2f}")

Como vamos a repetir este cálculo con cada canción que prueben, vale la pena llevarlo a una función.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `prediccion.py`</strong>
  Implementen en <code>src/radio_fcfm/prediccion.py</code>:

  <ul>
    <li><code>predecir_animo(modelo, letras)</code>: recibe un <code>Pipeline</code> ya ajustado y una lista de letras. Limpia cada letra con <code>limpiar_letra</code> y devuelve un <code>DataFrame</code> de pandas con una fila por letra, en el mismo orden, y tres columnas: <code>animo</code> (la clase de mayor probabilidad), <code>franja</code> (su franja de programación, con <code>a_franja</code>) y <code>confianza</code> (la probabilidad de la clase elegida, entre 0 y 1).</li>
  </ul>
</div>

Usemos la función para guardar cada canción que prueben en una tabla. Para agregar otra, cambien el título y la letra de la celda de `letra` y vuelvan a ejecutar las celdas desde ahí: cada canción queda como una fila de la tabla. Si repiten un título, esa fila se reemplaza.

In [ ]:
from src.radio_fcfm.prediccion import predecir_animo

resultado = predecir_animo(modelo_recargado, [letra]).iloc[0]
print(f"La función coincide con el notebook: {resultado['animo'] == animo}")

if "probadas" not in globals():  # la tabla se conserva entre ejecuciones
    probadas = {}

probadas[titulo] = resultado

filas = []
for nombre, fila in probadas.items():
    filas.append({"canción": nombre, **fila.to_dict()})
pl.DataFrame(filas).with_columns(pl.col("confianza").round(2))

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Ejecutar <code>uv run pytest -m etapa4</code>. Con la letra de ejemplo, la línea de la función debe imprimir <code>True</code> y la tabla debe tener una fila con el ánimo <code>happy</code>, la franja <code>mañana</code> y una confianza entre 0 y 1.
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Lo que el modelo entiende de una canción</strong>
  1. Prueben al menos tres canciones en inglés que conozcan, de ánimos distintos. ¿Coinciden las predicciones con lo que esperaban? ¿En cuál dudó más el modelo (la confianza más baja), y qué palabras de la letra, de las que vieron en la 3.4, podrían explicarlo?
  2. ¿La `confianza` es la probabilidad de que la predicción sea correcta? ¿Qué le advierten a Gaspar?
</div>

`Escriban sus respuestas aquí:`

### 4.5 — La recomendación para Gaspar (0,3 puntos)

Gaspar espera su consejo. Reúnan lo que midieron, que son los modelos candidatos con la misma validación, el resultado en el test y lo que dijo Jev, y decidan qué le recomendarían hacer. Qué se puede esperar de la franja de la mañana ya lo medimos en la 3.3.

In [ ]:
print(
    tabla_decision[
        ["tags.mlflow.runName", "metrics.promedio", "metrics.desviacion"]
    ]
    .round(3)
    .to_string(index=False)
)

In [ ]:
print(
    f"Test: F1 macro = {f1_test:.3f} | "
    f"acierto por franja = {franja_test:.3f}"
)
print(
    f"Jev: F1 macro = {f1_jev:.3f} | "
    f"acierto por franja = {franja_jev:.3f}"
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ La recomendación final</strong>
  1. Con los números de este notebook, ¿se puede conocer el ánimo de una canción solo a partir de su letra, que es lo que Gaspar quería saber? Justifiquen con las métricas.
  2. Con los mismos números, ¿qué le recomendarían hacer a Gaspar: dejar funcionando alguno de los modelos, probar otra cosa o esperar a tener más información? Digan también qué pueden prometerle. Justifiquen citando los runs de la tabla, el test, las métricas por franja y lo que cuesta cada alternativa, no una intuición.
  3. ¿Qué límites del modelo le dirían, para que no confíe más de lo que corresponde?
  4. ¿Qué próximos pasos le sugerirían si Gaspar quisiera mejorar la precisión y el accuracy? ¿Qué otros modelos podrían probar y qué otras variables, además de las palabras de la letra, podrían usar? Para cada propuesta, digan qué esperan y por qué.
</div>

`Escriban sus respuestas aquí:`

---

Las dudas se resuelven en el foro del curso en U-Cursos, y la entrega, con la Pull Request abierta, vence el domingo siguiente a esta sesión a las 23:59.